# Exp 7 (v2) — does a causal transformer share machinery across serializations of the same tree?

v2 after the first E1/E2 run. **Same grammar (`build_grammar(seed=0)`), same checkpoints, same alignment; what changed is the instrument.**

**What the first run showed.** Identity patch exact (plumbing fine). But at levels 4-5, whole-residual patches from the WRONG tree hurt about as much as patches with the RIGHT symbol (F ~ C0), so the contrast could not see NT identity, and the S values (down to -6) came from denominators ~0.03. E2: the A-vs-B level-6 control flagged ~80% of heads as "differing" although span content is identical, so the offset-profile comparison was uninterpretable. The ablation table (order-specific heads) was the only clean signal.

**v2 changes**
- **E1 ancestor-matched donors:** donor shares the recipient's ancestor symbol chain and canonical child index; only the finished span's symbol differs (`same_nt_m` vs `diff_nt_m`). Recipients lacking either donor are dropped so both modes cover the same samples.
- **E1b code swap (new, the main E1 measurement now):** transplant only the NT-identity component (token-out class-mean code), same symbol, leave context untouched. Ceiling = re-estimated code from the same language; floor = junk of matched norm; validity = swap-to-another-symbol must beat junk (the model actually reads this code). Limit: linear mean-difference code only.
- **Indices need range:** denominator must exceed max(0.05 nats, 3 SE) or the index is `nan`; 5-95% bootstrap intervals are printed. Layers moved to 1-7, 9 (8+ was dead).
- **E2:** A-vs-A' floor is used when the 4-language model exists; otherwise the level-6 A/B control is evaluated and the profile comparison is flagged invalid if it fails. Head ablation is now per-tree paired, with an order-specific-head test (|diff| >= 0.02 nats, |z| >= 4).
- **E4:** `run_e4.py` (separate file) builds the 4-language pool in parallel and launches training; the notebook cell does the same sequentially.
- **Plumbing assert extended:** `ident` and `cb:self` (h - d + d) must both give ~0 damage.

**Honesty note.** The E1 instrument was revised after seeing A->C numbers. Set thresholds from control rows only and confirm on seeds not used for setting them (see the prereg cell).

Other departures from the handoff, unchanged from v1: nb6's models are 12 layers (not 2) with two d=96 families (big-pool `n_head=4`, single-point `n_head=12`; `n_head` is set in `CKPTS`); `linearize_with_paths` supplies alignment via a language-independent instance id; under L_C the span END is a different leaf than in L_A, so alignment is by NT instance; eval is fp32 with TF32 off.


## 0. Switches, checkpoints, config

In [4]:
import os, sys, math, json, time, pickle, itertools, subprocess
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

# ============================== SWITCHES ==============================
RUN_E1 = True              # interchange intervention, eval only, existing checkpoints   <- run first
RUN_E2 = True              # attention-offset profiles + head ablation, eval only
RUN_E4_DATAGEN = True
RUN_E4_LAUNCH  = True    # launches the A' training run in the background (GPU, ~90K steps). Flip deliberately.

DEVICE = torch.device(os.environ.get("EXP_DEVICE", "cuda:2" if torch.cuda.is_available() else "cpu"))
torch.backends.cuda.matmul.allow_tf32 = False      # eval in fp32/no TF32: damage differences are small, bf16/TF32 noise would swamp them
torch.backends.cudnn.allow_tf32 = False
print("device:", DEVICE)

# ============================== CHECKPOINTS ==============================
# n_layer is read from the checkpoint; n_head CANNOT be inferred, so it is stated here. nb6 has two d=96 families:
#   bp_h4_*  : big-pool (500K trees), n_head=4,  seeds 0/7/13, 90K steps   -> the family the nb6 conclusions rest on
#   h12_*    : single-point check, n_head=12, seeds 0 / 7 (seed-7 dir: exp6_single_point_check_seed2)
CKPTS = {
    "bp_h4_d96_s0":  dict(path="./exp6_bigpool_h4/shared_d96_h4_seed0.pt",  n_head=4),
    "bp_h4_d96_s7":  dict(path="./exp6_bigpool_h4/shared_d96_h4_seed7.pt",  n_head=4),
    "bp_h4_d96_s13": dict(path="./exp6_bigpool_h4/shared_d96_h4_seed13.pt", n_head=4),
    "h12_d96_s0":    dict(path="./exp6_single_point_check/shared_d96.pt",       n_head=12),
    "h12_d96_s7":    dict(path="./exp6_single_point_check_seed2/shared_d96.pt", n_head=12),
    "e4_a2_d96_s0":  dict(path="./exp7_e4/shared4_d96_h4_seed0.pt",         n_head=4),   # produced by section E4 below
}
EVAL_MODELS = ["bp_h4_d96_s0", "bp_h4_d96_s7", "bp_h4_d96_s13"]      # models E1/E2 loop over (missing files are skipped, not fatal)

# ============================== EVAL SET ==============================
N_EVAL_TREES = 600
EVAL_SEED = 999            # nb6 used 999 / 300 trees. Rows are one tree each (BOS + leaves + 0-padding), NOT packed streams.

# ============================== E1 ==============================
LEVELS_E1 = (4, 5, 6)
LAYERS_E1 = (1, 2, 3, 4, 5, 6, 7, 9)   # block-OUTPUT indices (0..11). First run: layers 8+ had no dynamic range, so coverage is early/mid. 11 only changes the readout.
N_REP     = 4              # random recipient instance per tree per rep (reps reuse trees -> reported SEs are optimistic)
WINDOW    = 8              # score CE at positions p .. p+WINDOW-1 after the patched span end p (inside the tree)
PAIRS_E1  = [("L_A", "L_C"), ("L_C", "L_A"), ("L_A", "L_B"), ("L_B", "L_A"), ("L_B", "L_C"), ("L_C", "L_B")]   # (donor, recipient)
MIN_FLOOR_RANGE = 0.05     # nats (raised from 0.02: first run produced S values like -6 from denominators ~0.03). An index is also nan unless
                           # its denominator exceeds 3 standard errors. Still a placeholder until frozen in the prereg cell.
# ============================== E2 ==============================
LEVELS_E2 = (4, 5, 6)
K_MAX = 48                 # offset bins 0..K_MAX-1, bin K_MAX = "far"; the BOS key is accounted separately
N_SPLITS = 50              # random tree half-splits for the within-language floor


device: cuda:2


## 1. Grammar, indexed trees, position alignment (torch-free) + self-test

In [5]:
# ---------------- grammar + indexed trees + alignment (torch-free) ----------------
import random, zlib, numpy as np
from collections import defaultdict

LEVELS = 6; NT_SIZE = 4; TERM_SIZE = 4
MIN_DEGREE, MAX_DEGREE, RULE_LEN_CHOICES = 2, 4, [2, 3]
SEQ_LEN = 256
GRAMMAR_SEED = 0            # nb6's check_rules = build_grammar(seed=0): MUST match what the checkpoints were trained on

LANGS = {                    # order matters: BOS ids are TERM_SIZE + position, so 3-lang models keep ids 4,5,6
    "L_A":  set(),
    "L_B":  {4, 5},
    "L_C":  set(range(1, LEVELS + 1)),
    "L_A2": set(),           # E4 duplicate anchor: same order as L_A, different BOS tag (only exists in 4-lang models)
}
BOS_ALL = {n: TERM_SIZE + i for i, n in enumerate(LANGS)}

def build_grammar(seed):                       # verbatim from nb6
    rng = random.Random(seed)
    symbols_by_level = {1: [(1, 0)]}
    for l in range(2, LEVELS + 1):
        symbols_by_level[l] = [(l, i) for i in range(NT_SIZE)]
    terminals = list(range(TERM_SIZE))
    rules = defaultdict(list)
    for l in range(1, LEVELS + 1):
        child_level = l + 1
        child_symbols = terminals if child_level == LEVELS + 1 else symbols_by_level[child_level]
        for sym in symbols_by_level[l]:
            degree = rng.randint(MIN_DEGREE, MAX_DEGREE)
            seen = set(); tries = 0
            while len(rules[sym]) < degree and tries < 200:
                tries += 1
                rule_len = rng.choice(RULE_LEN_CHOICES)
                rhs = tuple(rng.choice(child_symbols) for _ in range(rule_len))
                if rhs in seen: continue
                seen.add(rhs); rules[sym].append(rhs)
    return rules, symbols_by_level

def sample_tree(rules, rng, symbol=(1, 0), level=1):          # verbatim from nb6 (used for E4 training data)
    rhs = rng.choice(rules[symbol]); children = []
    for child_sym in rhs:
        if level == LEVELS: children.append(child_sym)
        else: children.append(sample_tree(rules, rng, symbol=child_sym, level=level + 1))
    return (symbol, children)

def linearize(node, reverse_levels):                          # verbatim from nb6
    if isinstance(node, int): return [node]
    (level, idx), children = node
    order = list(reversed(children)) if level in reverse_levels else children
    tokens = []
    for c in order: tokens.extend(linearize(c, reverse_levels))
    return tokens

# --- indexed version. ONE definition (nb6 redefines these three times with different tuple shapes). ---
# internal node = (symbol=(level, idx), children, inst);  leaf = ("LEAF", token, leaf_id)
# inst = preorder counter, assigned in the CANONICAL tree, so it is language-independent: the same NT instance
# carries the same inst id in L_A, L_B, L_C. That is the alignment key for E1.
def sample_tree_indexed(rules, rng, symbol=(1, 0), level=1, _leaf=None, _inst=None):
    if _leaf is None: _leaf = [0]
    if _inst is None: _inst = [0]
    inst = _inst[0]; _inst[0] += 1
    rhs = rng.choice(rules[symbol]); children = []
    for child_sym in rhs:
        if level == LEVELS:
            children.append(("LEAF", child_sym, _leaf[0])); _leaf[0] += 1
        else:
            children.append(sample_tree_indexed(rules, rng, child_sym, level + 1, _leaf, _inst))
    return (symbol, children, inst)

def linearize_with_paths(node, reverse_levels, _path=()):
    """-> list of (token, canonical_leaf_id, path) in THIS language's order; path = ((level, idx, inst), ...) root->parent-of-leaf"""
    if node[0] == "LEAF":
        return [(node[1], node[2], _path)]
    (level, idx), children, inst = node
    order = list(reversed(children)) if level in reverse_levels else children
    new_path = _path + ((level, idx, inst),)
    out = []
    for c in order: out.extend(linearize_with_paths(c, reverse_levels, new_path))
    return out

def _seed(*a):                       # stable across processes (builtin string hashing is salted per process)
    return zlib.crc32(repr(a).encode())

def build_eval_core(rules, n_trees, seed, seq_len=SEQ_LEN):
    """Same trees, rendered in every language. Row layout: [BOS, leaf tokens..., 0-padding]. Pure python/numpy.
    SE[lang][level][(t, inst)] = (pos_first, pos_last, nt_idx, n_leaves): the contiguous span of that NT instance
    in this language's row. pos_last = the span END = where the model has just finished reading the constituent."""
    rng = random.Random(seed); cap = seq_len - 1
    trees = []
    while len(trees) < n_trees:
        t = sample_tree_indexed(rules, rng)
        if len(linearize_with_paths(t, set())) <= cap: trees.append(t)
    core = dict(n_trees=n_trees, seq_len=seq_len, tokens={}, nreal={}, SE={}, leafpos={})
    for lang, rev in LANGS.items():
        toks = np.zeros((n_trees, seq_len), np.int64); nreal = np.zeros(n_trees, np.int64)
        se = {l: {} for l in range(1, LEVELS + 1)}
        lp = []                                              # lp[t][leaf_id] = position in this language's row
        for t, tree in enumerate(trees):
            lin = linearize_with_paths(tree, rev)
            toks[t, 0] = BOS_ALL[lang]; nreal[t] = len(lin) + 1; d = {}
            for pos, (tok, lid, path) in enumerate(lin):
                p = pos + 1; toks[t, p] = tok; d[lid] = p
                for (lvl, idx, inst) in path:
                    e = se[lvl].get((t, inst))
                    if e is None: se[lvl][(t, inst)] = [p, p, idx, 1]
                    else: e[1] = p; e[3] += 1
            lp.append(d)
        core["tokens"][lang] = toks; core["nreal"][lang] = nreal
        core["SE"][lang] = {l: {k: tuple(v) for k, v in se[l].items()} for l in se}
        core["leafpos"][lang] = lp
    # language-independent donor pools: POOL[level] = list of (t, inst, nt_idx)
    core["POOL"] = {l: [(t, inst, v[2]) for (t, inst), v in core["SE"]["L_A"][l].items()] for l in range(1, LEVELS + 1)}
    core["trees"] = trees
    # ancestor-path metadata (language-independent): META[level][(t, inst)] = (chain, child_idx)
    #   chain = symbol idx of the ancestors at levels 1..level-1 ; child_idx = canonical position inside the parent's rule
    META = {l: {} for l in range(1, LEVELS + 1)}; BY = defaultdict(lambda: defaultdict(list))
    def walk(node, chain, cidx, t):
        if node[0] == "LEAF": return
        (level, idx), children, inst = node
        META[level][(t, inst)] = (chain, cidx); BY[(level, chain, cidx)][idx].append((t, inst))
        for j, c in enumerate(children): walk(c, chain + (idx,), j, t)
    for t, tree in enumerate(trees): walk(tree, (), 0, t)
    core["META"] = META; core["BY"] = BY
    return core

def alignment_selftest(core):
    """Hard invariants of the E1 position alignment. Run before trusting any patching number."""
    SE = core["SE"]; n = core["n_trees"]
    # 1. L_C is the exact reverse of L_A on terminals
    for t in range(min(n, 50)):
        a = core["tokens"]["L_A"][t, 1:core["nreal"]["L_A"][t]]; c = core["tokens"]["L_C"][t, 1:core["nreal"]["L_C"][t]]
        assert (a[::-1] == c).all(), "L_C is not reverse(L_A)"
    # 2. L_A2 renders identically to L_A apart from BOS
    assert (core["tokens"]["L_A"][:, 1:] == core["tokens"]["L_A2"][:, 1:]).all()
    assert (core["tokens"]["L_A"][:, 0] != core["tokens"]["L_A2"][:, 0]).all()
    # 3. every NT span is a contiguous interval, same size in every language, same NT symbol, same leaf SET
    for lang in LANGS:
        for l in range(1, LEVELS + 1):
            for key, (f, la, idx, cnt) in SE[lang][l].items():
                assert la - f + 1 == cnt, f"span not contiguous: {lang} level {l} {key}"
                f0, la0, idx0, cnt0 = SE["L_A"][l][key]
                assert cnt == cnt0 and idx == idx0
    # 4. span END is a different leaf under reversal: first leaf (A order) of a level-4 span == last position in C
    t, inst = next(iter(k for k in SE["L_A"][4]))
    inv = {p: lid for lid, p in core["leafpos"]["L_A"][t].items()}; invc = {p: lid for lid, p in core["leafpos"]["L_C"][t].items()}
    fa, la_, _, _ = SE["L_A"][4][(t, inst)]; fc, lc, _, _ = SE["L_C"][4][(t, inst)]
    assert inv[fa] == invc[lc], "span start in A must be span end in C (full reversal)"
    # 5. EXPECTED-SHARED CONTROL: level-6 internal order is identical in A and B (B only reverses levels 4,5)
    for t in range(min(n, 50)):
        for (tt, inst), (f, la, idx, cnt) in SE["L_A"][6].items():
            if tt != t: continue
            fb, lb, _, _ = SE["L_B"][6][(t, inst)]
            ordA = [lid for lid, p in sorted(core["leafpos"]["L_A"][t].items(), key=lambda x: x[1]) if f <= p <= la]
            ordB = [lid for lid, p in sorted(core["leafpos"]["L_B"][t].items(), key=lambda x: x[1]) if fb <= p <= lb]
            assert ordA == ordB, "level-6 span order differs between A and B"
    print("alignment self-test PASSED (reverse(A)==C, A2==A, spans contiguous, span-start(A)==span-end(C), level-6 A==B)")

def select_recipients(core, lang, level, rep, seed=0, min_next=1):
    """One NT instance per tree (level `level`), uniformly among instances whose span END has a next token in `lang`.
    -> list of (t, inst, nt_idx). Deterministic in (seed, level, rep, lang)."""
    rng = random.Random(_seed(seed, level, rep, lang))
    by_tree = defaultdict(list)
    for (t, inst), (f, la, idx, cnt) in core["SE"][lang][level].items():
        if la <= core["nreal"][lang][t] - 1 - min_next: by_tree[t].append((inst, idx))
    out = []
    for t in range(core["n_trees"]):
        if by_tree[t]:
            inst, idx = rng.choice(sorted(by_tree[t])); out.append((t, inst, idx))
    return out

def draw_donors(core, level, recs, mode, rep, seed=0):
    """Language-independent donor instances (t', inst') for each recipient (t, inst, nt):
       same_inst: the SAME instance of the SAME tree          (t'=t, inst'=inst)
       same_nt  : another tree, a level-`level` NT with the SAME symbol   (identity right, context wrong)
       diff_nt  : another tree, a level-`level` NT with a DIFFERENT symbol (identity wrong)"""
    if mode == "same_inst": return [(t, inst) for (t, inst, nt) in recs]
    rng = random.Random(_seed(seed, level, rep, "donor", mode)); pool = core["POOL"][level]; out = []
    for (t, inst, nt) in recs:
        while True:
            t2, i2, nt2 = pool[rng.randrange(len(pool))]
            if t2 != t and ((nt2 == nt) if mode == "same_nt" else (nt2 != nt)): out.append((t2, i2)); break
    return out


def draw_donors_matched(core, level, recs, rep, seed=0):
    """ANCESTOR-MATCHED donors. Donor instance (t', inst') comes from another tree and has the SAME ancestor symbol chain
    (levels 1..level-1) and the SAME canonical child index as the recipient instance, so the open parse state is matched
    and only the finished span's own symbol differs between the two modes:
        same_nt_m : donor span has the same symbol as the recipient's        diff_nt_m : a different symbol
    A recipient is dropped when either mode has no candidate, so both modes always cover the same recipients.
    -> (kept_recs, {"same_nt_m": [...], "diff_nt_m": [...]})"""
    rng = random.Random(_seed(seed, level, rep, "matched")); kept, same, diff = [], [], []
    for (t, inst, nt) in recs:
        chain, cidx = core["META"][level][(t, inst)]; by = core["BY"][(level, chain, cidx)]
        s_c = [x for x in by.get(nt, []) if x[0] != t]
        d_c = [x for n2, lst in by.items() if n2 != nt for x in lst if x[0] != t]
        if not s_c or not d_c: continue
        kept.append((t, inst, nt)); same.append(s_c[rng.randrange(len(s_c))]); diff.append(d_c[rng.randrange(len(d_c))])
    return kept, {"same_nt_m": same, "diff_nt_m": diff}


## 2. Model, loader, eval helpers

In [6]:
# ---- model: verbatim from nb6 (TinyRotaryGPT, RoPE on q/k). forward returns (logits, loss). ----
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)
    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head; self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd); self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd); self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln1(x)); x = x + self.mlp(self.ln2(x)); return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0 and (n_embd // n_head) % 2 == 0, "head_dim must be even for RoPE"
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks: x = blk(x)
        logits = self.head(self.ln_f(x)); loss = None
        if targets is not None: loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

def load_model(path, n_head, seq_len=SEQ_LEN):
    """vocab, d_model, n_layer are read off the checkpoint. 3-lang models (vocab 7) -> A,B,C; 4-lang (vocab 8) -> A,B,C,A2."""
    ck = torch.load(path, map_location="cpu", weights_only=False); sd = ck["model_state"]
    vocab, d = sd["tok_emb.weight"].shape
    n_layer = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))
    m = TinyRotaryGPT(vocab, d, n_layer, n_head, seq_len + 1); m.load_state_dict(sd); m.to(DEVICE).eval()
    langs = list(LANGS)[: vocab - TERM_SIZE]
    tail = ck["losses"][-1000:]
    print(f"  {path}: vocab={vocab} d={d} layers={n_layer} heads={n_head} step={ck['step']} tail-1000 loss={sum(tail)/len(tail):.4f} langs={langs}")
    return m, langs

@torch.no_grad()
def hidden_states(model, tokens, layers, bs=128):
    acts = {l: [] for l in layers}
    hs = [model.blocks[l].register_forward_hook(lambda m, i, o, l=l: acts[l].append(o.detach())) for l in layers]
    try:
        for i in range(0, tokens.shape[0], bs): model(tokens[i:i + bs])
    finally:
        for h in hs: h.remove()
    return {l: torch.cat(acts[l]) for l in layers}                      # {layer: (N, T, d)} on DEVICE

def _ce(logits, x):
    return F.cross_entropy(logits[:, :-1].reshape(-1, logits.shape[-1]).float(), x[:, 1:].reshape(-1),
                           reduction="none").view(x.shape[0], -1)         # (B, T-1); position p predicts token p+1

@torch.no_grad()
def mean_ce_tree(model, tokens, nreal, bs=128, pre_hook_layer=None, pre_hook=None):
    """Mean next-token CE over in-tree targets only (padding excluded). Optional forward-pre-hook on one module."""
    ar = torch.arange(tokens.shape[1] - 1, device=DEVICE); tot = 0.0; cnt = 0.0
    for i in range(0, tokens.shape[0], bs):
        x = tokens[i:i + bs]; nr = nreal[i:i + bs]; h = None
        if pre_hook is not None: h = pre_hook_layer.register_forward_pre_hook(pre_hook)
        try: logits, _ = model(x)
        finally:
            if h is not None: h.remove()
        m = (ar[None, :] <= (nr - 2)[:, None]).float(); tot += (_ce(logits, x) * m).sum().item(); cnt += m.sum().item()
    return tot / cnt

def make_eval_tensors(core):
    tok = {lg: torch.from_numpy(core["tokens"][lg]).to(DEVICE) for lg in LANGS}
    nreal = {lg: torch.from_numpy(core["nreal"][lg]).to(DEVICE) for lg in LANGS}
    return tok, nreal


## 3. E1 / E1b — interchange and code-swap (eval only, existing checkpoints)

In [8]:
# =============================== E1 (v2): interchange intervention ===============================
# damage = patched CE - clean CE over the next WINDOW tokens (nats); bigger = worse. Patch site = block-k OUTPUT at a span END.
#
# WHY v2. First run: whole-residual patches from the WRONG tree hurt about as much as patches with the RIGHT symbol (F ~ C0 at
# levels 4-5), so the contrast could not see NT identity. Likely cause: the span-end residual is dominated by open-parse state, and
# next-token damage tracks parent context, not the finished child's symbol. Two fixes:
#
#  (1) ANCESTOR-MATCHED whole-residual patches: donors share the recipient's ancestor symbol chain and canonical child index; only
#      the finished span's symbol differs (same_nt_m vs diff_nt_m). Index  S_nt = 1 - (XN_m - C0_m)/(F_m - C0_m).
#
#  (2) E1b "CODE SWAP": transplant only the NT-identity COMPONENT, leaving context in place. delta_X[s] = class-mean residual of symbol s
#      in language X, after removing per-(current,previous token) means (token-out) and centering. Class means are estimated on tree
#      tertiles; for a recipient in tertile p, the code is REMOVED with tertile p+1's table and REPLACED from tertile p+2's table.
#        cb:within  h - d_R[p+1][nt] + d_R[p+2][nt]   same language, same symbol, re-estimated code      (ceiling: estimation noise only)
#        cb:R<-D    h - d_R[p+1][nt] + d_D[p+2][nt]   donor language's code for the SAME symbol           (the sharing test)
#        cb:junk    h - d_R[p+1][nt] + random direction with norm |d_R[p+2][nt]|                         (floor: code replaced by junk)
#        cb:erase   h - d_R[p+1][nt]
#        cb:swap    h - d_R[p+1][nt] + d_R[p+2][other symbol]    (validity: if swap ~ junk the model does not read this code at all)
#        cb:self    h - d + d  ==  h   (plumbing assert)
#      S_code = 1 - (cross - within)/(junk - within). Because the SAME symbol is transplanted, a shared code costs ~nothing; no
#      context is touched. Limit: this tests the linear mean-difference code only; a nonlinear/distributed code can still be shared.

@torch.no_grad()
def window_ce(model, tokens, rec_t, rec_pos, nreal, W, k=None, vecs=None, bs=128):
    n = rec_t.shape[0]; win, nxt = [], []; ar = torch.arange(tokens.shape[1] - 1, device=DEVICE)
    for i in range(0, n, bs):
        t = rec_t[i:i + bs]; p = rec_pos[i:i + bs]; x = tokens[t]; B = x.shape[0]; h = None
        if vecs is not None:
            v = vecs[i:i + bs]; ab = torch.arange(B, device=DEVICE)
            def hook(m, inp, out, v=v, p=p, ab=ab):
                out = out.clone(); out[ab, p] = v.to(out.dtype); return out
            h = model.blocks[k].register_forward_hook(hook)
        try: logits, _ = model(x)
        finally:
            if h is not None: h.remove()
        ce = _ce(logits, x)
        m = ((ar[None, :] >= p[:, None]) & (ar[None, :] < p[:, None] + W) & (ar[None, :] <= (nreal[t] - 2)[:, None])).float()
        win.append((ce * m).sum(1) / m.sum(1).clamp(min=1)); nxt.append(ce[torch.arange(B, device=DEVICE), p])
    return torch.cat(win), torch.cat(nxt)

@torch.no_grad()
def build_code_tables(core, H, tok, langs, levels, layers):
    """tables[(lang, level, layer)] -> (3, NT_SIZE, d): token-out class-mean code per symbol, per tree-tertile (t % 3)."""
    SE = core["SE"]; tables = {}
    for lg in langs:
        for l in levels:
            items = list(SE[lg][l].items())
            ts = torch.tensor([k[0] for k, _ in items], device=DEVICE); ps = torch.tensor([v[1] for _, v in items], device=DEVICE)
            nts = torch.tensor([v[2] for _, v in items], device=DEVICE)
            key = tok[lg][ts, ps] * 8 + tok[lg][ts, ps - 1]                          # (current, previous) token pair; ids < 8
            for k in layers:
                X = H[lg][k][ts, ps]; tab = torch.zeros(3, NT_SIZE, X.shape[1], device=DEVICE)
                for s in range(3):
                    m = (ts % 3) == s; Xs, ks, ns = X[m], key[m], nts[m]
                    uk, inv = torch.unique(ks, return_inverse=True)
                    sums = torch.zeros(len(uk), X.shape[1], device=DEVICE).index_add_(0, inv, Xs)
                    cnt = torch.bincount(inv, minlength=len(uk)).clamp(min=1).unsqueeze(1)
                    Xt = Xs - (sums / cnt)[inv]
                    for c in range(NT_SIZE):
                        mc = ns == c
                        if mc.any(): tab[s, c] = Xt[mc].mean(0)
                tables[(lg, l, k)] = tab
    return tables

def run_E1(model, langs, core, tok, nreal, pairs=PAIRS_E1, levels=LEVELS_E1, layers=LAYERS_E1, n_rep=N_REP, W=WINDOW):
    pairs = [(d, r) for d, r in pairs if d in langs and r in langs]
    if "L_A2" in langs: pairs += [("L_A2", "L_A"), ("L_A", "L_A2")]              # E4 ceiling pair, only in the 4-language model
    recips = sorted({r for _, r in pairs}); donors_of = {r: [d for d, r2 in pairs if r2 == r] for r in recips}
    H = {lg: hidden_states(model, tok[lg], layers) for lg in langs}
    SE = core["SE"]; stats = {}
    for lg in langs:
        for l in levels:
            items = list(SE[lg][l].items())
            ts = torch.tensor([k[0] for k, _ in items], device=DEVICE); ps = torch.tensor([v[1] for _, v in items], device=DEVICE)
            for k in layers:
                X = H[lg][k][ts, ps]; stats[(lg, l, k)] = (X.mean(0), X.std(0))
    tables = build_code_tables(core, H, tok, langs, levels, layers)
    acc = {}
    for l in levels:
        for rep in range(n_rep):
            for R in recips:
                recs0 = select_recipients(core, R, l, rep)
                recs, dn_m = draw_donors_matched(core, l, recs0, rep)                 # recipients for which BOTH matched modes exist
                dn = {m: draw_donors(core, l, recs, m, rep) for m in ("same_inst", "same_nt", "diff_nt")}; dn.update(dn_m)
                rt = torch.tensor([r[0] for r in recs], device=DEVICE); rp = torch.tensor([SE[R][l][(r[0], r[1])][1] for r in recs], device=DEVICE)
                nts = torch.tensor([r[2] for r in recs], device=DEVICE); n = rt.shape[0]
                cw, cn = window_ce(model, tok[R], rt, rp, nreal[R], W)
                def dvec(D, mode, k):
                    dl = dn[mode]; dt = torch.tensor([x[0] for x in dl], device=DEVICE)
                    dp = torch.tensor([SE[D][l][x][1] for x in dl], device=DEVICE); return H[D][k][dt, dp]
                rem_s, rep_s = (rt + 1) % 3, (rt + 2) % 3
                for k in layers:
                    mu, sd = stats[(R, l, k)]; h = H[R][k][rt, rp]
                    g = torch.Generator(device=DEVICE); g.manual_seed(_seed(l, rep, R, k))
                    conds = {"ident": h, "noise": mu + sd * torch.randn(n, mu.shape[0], generator=g, device=DEVICE), "mean": mu.expand(n, -1)}
                    for mode in ("same_nt", "diff_nt", "same_nt_m", "diff_nt_m"): conds[f"{R}<-{R}:{mode}"] = dvec(R, mode, k)
                    for D in donors_of[R]:
                        for mode in ("same_inst", "same_nt", "same_nt_m"): conds[f"{R}<-{D}:{mode}"] = dvec(D, mode, k)
                    TR = tables[(R, l, k)]; own = TR[rem_s, nts]; code_in = TR[rep_s, nts]
                    junk = torch.randn(n, h.shape[1], generator=g, device=DEVICE); junk = junk / junk.norm(dim=1, keepdim=True) * code_in.norm(dim=1, keepdim=True)
                    other = (nts + torch.randint(1, NT_SIZE, (n,), generator=g, device=DEVICE)) % NT_SIZE
                    conds["cb:self"] = h - own + own; conds["cb:within"] = h - own + code_in; conds["cb:erase"] = h - own
                    conds["cb:junk"] = h - own + junk; conds["cb:swap"] = h - own + TR[rep_s, other]
                    for D in donors_of[R]: conds[f"cb:{R}<-{D}"] = h - own + tables[(D, l, k)][rep_s, nts]
                    for name, vecs in conds.items():
                        w, nx = window_ce(model, tok[R], rt, rp, nreal[R], W, k=k, vecs=vecs)
                        acc.setdefault((R, l, k, name), []).append(((w - cw).cpu(), (nx - cn).cpu()))
    res = {}
    for key, lst in acc.items():
        w = torch.cat([a for a, _ in lst]).numpy(); x = torch.cat([b for _, b in lst]).numpy()
        res[key] = dict(win=float(w.mean()), win_se=float(w.std(ddof=1) / len(w) ** 0.5), nxt=float(x.mean()), nxt_se=float(x.std(ddof=1) / len(x) ** 0.5), n=len(w), w=w, x=x)
    bad = {k: v["win"] for k, v in res.items() if k[3] in ("ident", "cb:self") and abs(v["win"]) > 1e-3}
    print(f"  identity-type patches (ident, cb:self): max |damage| = {max(abs(v['win']) for k, v in res.items() if k[3] in ('ident','cb:self')):.2e} nats")
    assert not bad, f"identity patches changed the CE (indexing/plumbing bug): {list(bad.items())[:3]}"
    return dict(res=res, pairs=pairs, donors_of=donors_of, recips=recips, levels=list(levels), layers=list(layers))

# ----------------------------- reporting -----------------------------
def _boot(fn, arrs, B=300, seed=0):
    n = len(next(iter(arrs.values()))); idx = np.random.RandomState(seed).randint(0, n, (B, n))
    s = fn({k: v[idx].mean(1) for k, v in arrs.items()}); return float(np.nanpercentile(s, 5)), float(np.nanpercentile(s, 95))

def _se(a): return float(a.std(ddof=1) / len(a) ** 0.5)
def _fmt(s, ci): return "  nan            " if s != s else f"{s:5.2f}[{ci[0]:5.2f},{ci[1]:5.2f}]"

def index_matched(res, R, l, k, D, min_range=MIN_FLOOR_RANGE):
    A = lambda n: res[(R, l, k, n)]["w"]
    F_, C0, XN = A(f"{R}<-{R}:diff_nt_m"), A(f"{R}<-{R}:same_nt_m"), A(f"{R}<-{D}:same_nt_m")
    rng = F_ - C0; ok = rng.mean() > max(min_range, 3 * _se(rng))
    S = 1 - (XN.mean() - C0.mean()) / rng.mean() if ok else float("nan")
    ci = _boot(lambda m: np.where(m["F"] - m["C"] > 0, 1 - (m["XN"] - m["C"]) / (m["F"] - m["C"]), np.nan), dict(F=F_, C=C0, XN=XN)) if ok else (float("nan"),) * 2
    return dict(S=S, ci=ci, range=float(rng.mean()), range_z=float(rng.mean() / _se(rng)), ok=ok, F=float(F_.mean()), C0=float(C0.mean()), XN=float(XN.mean()),
                X=float(A(f"{R}<-{D}:same_inst").mean()))

def index_code(res, R, l, k, D, min_range=MIN_FLOOR_RANGE):
    A = lambda n: res[(R, l, k, n)]["w"]
    Wn, J, Sw = A("cb:within"), A("cb:junk"), A("cb:swap"); X = A(f"cb:{R}<-{D}")
    rng = J - Wn; reads = (Sw - J).mean() > 3 * _se(Sw - J) and (Sw - J).mean() > 0
    ok = rng.mean() > max(min_range, 3 * _se(rng)) and reads
    S = 1 - (X.mean() - Wn.mean()) / rng.mean() if ok else float("nan")
    ci = _boot(lambda m: np.where(m["J"] - m["W"] > 0, 1 - (m["X"] - m["W"]) / (m["J"] - m["W"]), np.nan), dict(J=J, W=Wn, X=X)) if ok else (float("nan"),) * 2
    return dict(S=S, ci=ci, ok=ok, reads=bool(reads), within=float(Wn.mean()), junk=float(J.mean()), swap=float(Sw.mean()), erase=float(A("cb:erase").mean()),
                cross=float(X.mean()), reads_strength=float((Sw - J).mean()))

def report_E1(out, tag=""):
    res = out["res"]
    for R in out["recips"]:
        for l in out["levels"]:
            print(f"\n[{tag}] WHOLE-RESIDUAL, ancestor-matched | recipient {R}, level {l} | damage in nats over next {WINDOW} tokens")
            hdr = f"{'k':>2s} | {'noise':>5s} {'mean':>5s} | {'Fm':>5s} {'C0m':>5s} {'range z':>7s} |"
            for D in out["donors_of"][R]: hdr += f" {D+':X':>7s} {'XNm':>5s} {'S_nt_m [5%,95%]':>18s} |"
            print(hdr)
            for k in out["layers"]:
                g = lambda n: res[(R, l, k, n)]["win"]; row = None
                line = ""
                for D in out["donors_of"][R]:
                    r = index_matched(res, R, l, k, D)
                    if row is None: row = r; line = f"{k:>2d} | {g('noise'):5.3f} {g('mean'):5.3f} | {r['F']:5.3f} {r['C0']:5.3f} {r['range_z']:7.1f} |"
                    line += f" {r['X']:7.3f} {r['XN']:5.3f} {_fmt(r['S'], r['ci']):>18s} |"
                print(line)
    print("\nS_nt_m=1: cross-order same-symbol patch as harmless as within-order; 0: as harmful as wrong-symbol. nan = Fm-C0m not above max(MIN_FLOOR_RANGE, 3 SE).")
    print("If range z is ~0 at levels 4-5 the whole-residual instrument still cannot see NT identity -> rely on E1b below.")

def report_E1b(out, tag=""):
    res = out["res"]
    for R in out["recips"]:
        for l in out["levels"]:
            print(f"\n[{tag}] CODE SWAP (E1b) | recipient {R}, level {l} | damage in nats")
            hdr = f"{'k':>2s} | {'within':>6s} {'erase':>6s} {'junk':>6s} {'swap':>6s} {'reads?':>6s} |"
            for D in out["donors_of"][R]: hdr += f" {D+':cross':>9s} {'S_code [5%,95%]':>18s} |"
            print(hdr)
            for k in out["layers"]:
                line = ""
                for D in out["donors_of"][R]:
                    r = index_code(res, R, l, k, D)
                    if not line: line = f"{k:>2d} | {r['within']:6.3f} {r['erase']:6.3f} {r['junk']:6.3f} {r['swap']:6.3f} {('yes' if r['reads'] else 'NO'):>6s} |"
                    line += f" {r['cross']:9.3f} {_fmt(r['S'], r['ci']):>18s} |"
                print(line)
    print("\nS_code=1: donor language's code for the same symbol is as good as the recipient's own re-estimated code; 0: as bad as junk. nan = no dynamic range or the model does not read the code (swap ~ junk).")


In [5]:
core = build_eval_core(build_grammar(GRAMMAR_SEED)[0], N_EVAL_TREES, EVAL_SEED)
alignment_selftest(core)
tok, nreal = make_eval_tensors(core)
E1_OUT = {}
if RUN_E1:
    for name in EVAL_MODELS:
        if not os.path.exists(CKPTS[name]["path"]): print(f"[skip] {name}: {CKPTS[name]['path']} not found"); continue
        print(f"=== E1 on {name} ==="); model, langs = load_model(CKPTS[name]["path"], CKPTS[name]["n_head"])
        E1_OUT[name] = run_E1(model, langs, core, tok, nreal)
        report_E1(E1_OUT[name], tag=name); report_E1b(E1_OUT[name], tag=name)
        del model; torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
    os.makedirs("./exp7_out", exist_ok=True)
    pickle.dump({n: {k: v for k, v in o.items()} for n, o in E1_OUT.items()}, open("./exp7_out/e1_v2_results.pkl", "wb"))
else: print("RUN_E1 is False")


alignment self-test PASSED (reverse(A)==C, A2==A, spans contiguous, span-start(A)==span-end(C), level-6 A==B)
=== E1 on bp_h4_d96_s0 ===
  ./exp6_bigpool_h4/shared_d96_h4_seed0.pt: vocab=7 d=96 layers=12 heads=4 step=90000 tail-1000 loss=0.7973 langs=['L_A', 'L_B', 'L_C']
  identity-type patches (ident, cb:self): max |damage| = 3.18e-09 nats

[bp_h4_d96_s0] WHOLE-RESIDUAL, ancestor-matched | recipient L_A, level 4 | damage in nats over next 8 tokens
 k | noise  mean |    Fm   C0m range z |   L_C:X   XNm    S_nt_m [5%,95%] |   L_B:X   XNm    S_nt_m [5%,95%] |
 1 | 0.446 0.498 | 0.689 0.625     3.3 |   0.794 0.779 -1.41[-3.61,-0.67] |   0.771 0.778 -1.39[-3.41,-0.68] |
 2 | 0.350 0.395 | 0.543 0.500     2.6 |   0.644 0.622    nan             |   0.624 0.636    nan             |
 3 | 0.223 0.210 | 0.348 0.291     4.2 |   0.462 0.464 -2.02[-3.78,-1.31] |   0.447 0.444 -1.67[-3.24,-0.99] |
 4 | 0.178 0.112 | 0.256 0.187     5.8 |   0.439 0.447 -2.74[-4.04,-2.03] |   0.414 0.421 -2.37[-3.50,

## 4. E2 — attention-offset profiles + head ablation (eval only)

In [9]:
def attn_probs(attn, x):
    B, T, C = x.shape; H, hd = attn.n_head, attn.head_dim
    qkv = attn.qkv(x).view(B, T, 3, H, hd).permute(2, 0, 3, 1, 4); q, k = qkv[0], qkv[1]
    cos, sin = attn.rotary(T); q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
    s = (q @ k.transpose(-1, -2)) / math.sqrt(hd)
    s = s.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), 1), float("-inf"))
    return s.float().softmax(-1)                                                   # (B, H, T, T)

@torch.no_grad()
def offset_profiles(model, core, tok, lang, level, bs=64):
    """mass[t, layer, head, bin]: attention mass from level-`level` span-end queries of tree t, binned by offset (query - key);
    BOS key excluded from the bins and returned separately; bin K_MAX = offsets >= K_MAX."""
    L = len(model.blocks); H = model.blocks[0].attn.n_head; N = tok.shape[0]
    mass = torch.zeros(L, N, H, K_MAX + 1, device=DEVICE); bosm = torch.zeros(L, N, H, device=DEVICE)
    items = list(core["SE"][lang][level].items())
    qt_all = torch.tensor([k[0] for k, _ in items]); qp_all = torch.tensor([v[1] for _, v in items]); ctx = {}
    def make_pre(l):
        def pre(mod, args):
            P = attn_probs(mod, args[0]); qb, qp, qt = ctx["qb"], ctx["qp"], ctx["qt"]
            rows = P[qb, :, qp, :]                                                  # (Q, H, T)
            ar = torch.arange(rows.shape[-1], device=DEVICE); off = qp[:, None] - ar[None, :]
            valid = ((off >= 0) & (ar[None, :] >= 1)).float(); bins = off.clamp(0, K_MAX)
            m = torch.zeros(rows.shape[0], H, K_MAX + 1, device=DEVICE).scatter_add_(2, bins[:, None, :].expand(-1, H, -1), rows * valid[:, None, :])
            mass[l].index_add_(0, qt, m); bosm[l].index_add_(0, qt, rows[:, :, 0])
        return pre
    hs = [model.blocks[l].attn.register_forward_pre_hook(make_pre(l)) for l in range(L)]
    try:
        for s in range(0, N, bs):
            sel = (qt_all >= s) & (qt_all < s + bs)
            if not sel.any(): continue
            ctx["qt"] = qt_all[sel].to(DEVICE); ctx["qp"] = qp_all[sel].to(DEVICE); ctx["qb"] = ctx["qt"] - s
            model(tok[s:s + bs])
    finally:
        for h in hs: h.remove()
    return dict(mass=mass.permute(1, 0, 2, 3).cpu().numpy(), bos=bosm.permute(1, 0, 2).cpu().numpy())

def _emd(p, q): return np.abs(np.cumsum(p - q, -1)).sum(-1)                         # 1-D earth mover's distance over offset bins
def _norm(mass, trees):
    m = mass[trees].sum(0); return m / np.clip(m.sum(-1, keepdims=True), 1e-12, None)   # (L, H, K+1)

def profile_distances(profA, profB, n_split=N_SPLITS, seed=0):
    """cross = mean EMD(A on half 1, B on half 2); floor = EMD(A on half 1, A on half 2). Same half sizes, so noise is matched.
    A head 'differs' if cross > the 95th percentile of its own within-language split floor."""
    rng = np.random.RandomState(seed); N = profA["mass"].shape[0]; cross, floor = [], []
    for _ in range(n_split):
        perm = rng.permutation(N); s1, s2 = perm[: N // 2], perm[N // 2:]
        a1, a2, b2 = _norm(profA["mass"], s1), _norm(profA["mass"], s2), _norm(profB["mass"], s2)
        cross.append(_emd(a1, b2)); floor.append(_emd(a1, a2))
    cross, floor = np.stack(cross), np.stack(floor)
    return dict(cross=cross.mean(0), cross_all=cross, floor_mean=floor.mean(0), floor95=np.percentile(floor, 95, axis=0))   # (L, H) / (S, L, H)

@torch.no_grad()
def head_mean_outputs(model, toks, nreals, bs=128):
    """Language-pooled mean of each head's pre-projection output over in-tree positions (the mean-ablation target).
    Pooled across languages on purpose: a per-language mean would partly bake the language into the ablation."""
    L = len(model.blocks); H = model.blocks[0].attn.n_head; hd = model.blocks[0].attn.head_dim
    sums = torch.zeros(L, H, hd, device=DEVICE); total = 0.0; ctx = {}
    def make(l):
        def pre(mod, args):
            y = args[0]; yy = y.view(y.shape[0], y.shape[1], H, hd)
            sums[l] += (yy * ctx["m"][:, :, None, None]).sum((0, 1))
        return pre
    hs = [model.blocks[l].attn.proj.register_forward_pre_hook(make(l)) for l in range(L)]
    try:
        for tk, nr in zip(toks, nreals):
            ar = torch.arange(tk.shape[1], device=DEVICE)
            for i in range(0, tk.shape[0], bs):
                ctx["m"] = (ar[None, :] < nr[i:i + bs][:, None]).float(); total += ctx["m"].sum().item()
                model(tk[i:i + bs])
    finally:
        for h in hs: h.remove()
    return sums / total

@torch.no_grad()
def ce_per_tree(model, tokens, nreal, bs=128, pre_hook_layer=None, pre_hook=None):
    ar = torch.arange(tokens.shape[1] - 1, device=DEVICE); out = []
    for i in range(0, tokens.shape[0], bs):
        x = tokens[i:i + bs]; nr = nreal[i:i + bs]; h = None
        if pre_hook is not None: h = pre_hook_layer.register_forward_pre_hook(pre_hook)
        try: logits, _ = model(x)
        finally:
            if h is not None: h.remove()
        m = (ar[None, :] <= (nr - 2)[:, None]).float(); out.append(((_ce(logits, x) * m).sum(1) / m.sum(1)).cpu())
    return torch.cat(out).numpy()

def head_ablation(model, core, tok, nreal, langs):
    L = len(model.blocks); H = model.blocks[0].attn.n_head; hd = model.blocks[0].attn.head_dim
    mu = head_mean_outputs(model, [tok[lg] for lg in langs], [nreal[lg] for lg in langs])
    base_t = {lg: ce_per_tree(model, tok[lg], nreal[lg]) for lg in langs}; N = len(base_t[langs[0]])
    dtree = {lg: np.zeros((L, H, N)) for lg in langs}                                  # per-tree CE change under ablation (paired with clean)
    for l in range(L):
        for h in range(H):
            def pre(mod, args, l=l, h=h):
                y = args[0].clone(); y[..., h * hd:(h + 1) * hd] = mu[l, h].to(y.dtype); return (y,)
            for lg in langs:
                dtree[lg][l, h] = ce_per_tree(model, tok[lg], nreal[lg], pre_hook_layer=model.blocks[l].attn.proj, pre_hook=pre) - base_t[lg]
    return dict(base={lg: float(base_t[lg].mean()) for lg in langs}, dce={lg: dtree[lg].mean(-1) for lg in langs}, dtree=dtree)

def ablation_specificity(dtree, a, b, z_min=4.0, effect_min=0.02):
    """Paired over trees (same trees, different rendering): d_t = dCE_b,t - dCE_a,t per head. A head is 'order-specific' if |mean d| >= effect_min
    and |z| >= z_min. -> (mean_diff (L,H), z (L,H), specific bool (L,H))"""
    d = dtree[b] - dtree[a]; m = d.mean(-1); se = d.std(-1, ddof=1) / np.sqrt(d.shape[-1]); z = m / np.clip(se, 1e-9, None)
    return m, z, (np.abs(m) >= effect_min) & (np.abs(z) >= z_min)

def _rank(a): return np.argsort(np.argsort(a)).astype(float)

def run_E2(model, langs, core, tok, nreal, levels=LEVELS_E2, comparisons=(("L_A", "L_C"), ("L_A", "L_B"), ("L_A", "L_A2"))):
    prof = {(lg, l): offset_profiles(model, core, tok[lg], lg, l) for lg in langs for l in levels}
    dist = {(a, b, l): profile_distances(prof[(a, l)], prof[(b, l)]) for a, b in comparisons if a in langs and b in langs for l in levels}
    abl = head_ablation(model, core, tok, nreal, langs)
    return dict(prof_bos={k: v["bos"].mean(0) for k, v in prof.items()}, dist=dist, abl=abl, levels=list(levels))

def report_E2(out, tag="", top_frac=0.25, n_show=12):
    abl = out["abl"]; langs = list(abl["dce"]); mx = np.max([abl["dce"][lg] for lg in langs], axis=0)       # importance = worst-case language
    matter = mx >= np.quantile(mx, 1 - top_frac)
    print(f"\n[{tag}] HEAD ABLATION (language-pooled mean ablation), CE change per language; baseline CE: " + ", ".join(f"{lg} {abl['base'][lg]:.3f}" for lg in langs))
    order = np.dstack(np.unravel_index(np.argsort(-mx, axis=None), mx.shape))[0][:n_show]
    print(f"  top {n_show} heads by max dCE   (layer,head)  " + "  ".join(f"d{lg[-1]}" for lg in langs))
    for l, h in order: print(f"    L{l:<2d}H{h}   " + "  ".join(f"{abl['dce'][lg][l, h]:+.3f}" for lg in langs))
    print("\n  ORDER-SPECIFIC HEADS (paired over trees; |mean diff| >= 0.02 nats and |z| >= 4).  A-vs-B is the nearer-order comparison, A-vs-A2 (if present) the null.")
    n_all = mx.size
    for b in [lg for lg in langs if lg != "L_A"]:
        m, z, sp = ablation_specificity(abl["dtree"], "L_A", b)
        big = np.abs(m) >= 0.02
        print(f"   A vs {b}: {int(sp.sum())}/{n_all} heads specific  ({int(big.sum())} with |diff|>=0.02)  | " +
              "  ".join(f"L{l}H{h}:{m[l, h]:+.3f}" for l, h in sorted(zip(*np.nonzero(sp)), key=lambda x: -abs(m[x]))[:6]))
    # ---- offset profiles: only interpretable if the control passes
    print("\n  OFFSET-PROFILE COMPARISON (A vs other). 'differs' = cross EMD > floor.")
    has_a2 = ("L_A", "L_A2", out["levels"][0]) in out["dist"]
    if has_a2: print("   floor = 95th percentile of EMD(A half, A' half): same order, different tag (the right null).")
    else:
        ctrl = out["dist"].get(("L_A", "L_B", 6))
        frac = float((ctrl["cross"] > ctrl["floor95"]).mean()) if ctrl else float("nan")
        print(f"   no A' model yet: floor = within-language split floor only. CONTROL: level-6 A-vs-B (identical span content) flags {frac:.0%} of heads.")
        if frac > 0.25: print("   >>> CONTROL FAILS: the split floor only captures sampling noise, not position/context differences. Profile comparison below is NOT interpretable. <<<")
    print(f"  {'cmp':>10s} {'lvl':>3s} | {'#differ':>7s} {'differ|matter':>13s} {'differ|other':>12s}")
    for (a, b, l), d in sorted(out["dist"].items(), key=lambda kv: (kv[0][1], kv[0][2])):
        if b == "L_A2": continue
        if has_a2: floor = np.percentile(out["dist"][("L_A", "L_A2", l)]["cross_all"], 95, axis=0)
        else: floor = d["floor95"]
        differs = d["cross"] > floor
        fm = differs[matter].mean() if matter.any() else float("nan"); fo = differs[~matter].mean() if (~matter).any() else float("nan")
        print(f"  {a[-1]}-vs-{b[-1]:>6s} {l:>3d} | {int(differs.sum()):>3d}/{differs.size:<3d} {fm:13.2f} {fo:12.2f}")
    print("  Caveat: attention at span ends is a thin slice of what a head does; identical offset profiles do not imply identical content.")

In [6]:
# =============================== E2: attention-offset profiles + head ablation (M1) ===============================
# For every head: how much attention mass sits at each relative offset, measured at the span ENDS of level-l NT instances,
# separately per language on the SAME trees. Then: do profiles differ across languages (vs a within-language split floor),
# and are the heads that differ the ones whose ablation matters? Single model only; never compare head indices across seeds.

def attn_probs(attn, x):
    B, T, C = x.shape; H, hd = attn.n_head, attn.head_dim
    qkv = attn.qkv(x).view(B, T, 3, H, hd).permute(2, 0, 3, 1, 4); q, k = qkv[0], qkv[1]
    cos, sin = attn.rotary(T); q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
    s = (q @ k.transpose(-1, -2)) / math.sqrt(hd)
    s = s.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), 1), float("-inf"))
    return s.float().softmax(-1)                                                   # (B, H, T, T)

@torch.no_grad()
def offset_profiles(model, core, tok, lang, level, bs=64):
    """mass[t, layer, head, bin]: attention mass from level-`level` span-end queries of tree t, binned by offset (query - key);
    BOS key excluded from the bins and returned separately; bin K_MAX = offsets >= K_MAX."""
    L = len(model.blocks); H = model.blocks[0].attn.n_head; N = tok.shape[0]
    mass = torch.zeros(L, N, H, K_MAX + 1, device=DEVICE); bosm = torch.zeros(L, N, H, device=DEVICE)
    items = list(core["SE"][lang][level].items())
    qt_all = torch.tensor([k[0] for k, _ in items]); qp_all = torch.tensor([v[1] for _, v in items]); ctx = {}
    def make_pre(l):
        def pre(mod, args):
            P = attn_probs(mod, args[0]); qb, qp, qt = ctx["qb"], ctx["qp"], ctx["qt"]
            rows = P[qb, :, qp, :]                                                  # (Q, H, T)
            ar = torch.arange(rows.shape[-1], device=DEVICE); off = qp[:, None] - ar[None, :]
            valid = ((off >= 0) & (ar[None, :] >= 1)).float(); bins = off.clamp(0, K_MAX)
            m = torch.zeros(rows.shape[0], H, K_MAX + 1, device=DEVICE).scatter_add_(2, bins[:, None, :].expand(-1, H, -1), rows * valid[:, None, :])
            mass[l].index_add_(0, qt, m); bosm[l].index_add_(0, qt, rows[:, :, 0])
        return pre
    hs = [model.blocks[l].attn.register_forward_pre_hook(make_pre(l)) for l in range(L)]
    try:
        for s in range(0, N, bs):
            sel = (qt_all >= s) & (qt_all < s + bs)
            if not sel.any(): continue
            ctx["qt"] = qt_all[sel].to(DEVICE); ctx["qp"] = qp_all[sel].to(DEVICE); ctx["qb"] = ctx["qt"] - s
            model(tok[s:s + bs])
    finally:
        for h in hs: h.remove()
    return dict(mass=mass.permute(1, 0, 2, 3).cpu().numpy(), bos=bosm.permute(1, 0, 2).cpu().numpy())

def _emd(p, q): return np.abs(np.cumsum(p - q, -1)).sum(-1)                         # 1-D earth mover's distance over offset bins
def _norm(mass, trees):
    m = mass[trees].sum(0); return m / np.clip(m.sum(-1, keepdims=True), 1e-12, None)   # (L, H, K+1)

def profile_distances(profA, profB, n_split=N_SPLITS, seed=0):
    """cross = mean EMD(A on half 1, B on half 2); floor = EMD(A on half 1, A on half 2). Same half sizes, so noise is matched.
    A head 'differs' if cross > the 95th percentile of its own within-language split floor."""
    rng = np.random.RandomState(seed); N = profA["mass"].shape[0]; cross, floor = [], []
    for _ in range(n_split):
        perm = rng.permutation(N); s1, s2 = perm[: N // 2], perm[N // 2:]
        a1, a2, b2 = _norm(profA["mass"], s1), _norm(profA["mass"], s2), _norm(profB["mass"], s2)
        cross.append(_emd(a1, b2)); floor.append(_emd(a1, a2))
    cross, floor = np.stack(cross), np.stack(floor)
    return dict(cross=cross.mean(0), cross_all=cross, floor_mean=floor.mean(0), floor95=np.percentile(floor, 95, axis=0))   # (L, H) / (S, L, H)

@torch.no_grad()
def head_mean_outputs(model, toks, nreals, bs=128):
    """Language-pooled mean of each head's pre-projection output over in-tree positions (the mean-ablation target).
    Pooled across languages on purpose: a per-language mean would partly bake the language into the ablation."""
    L = len(model.blocks); H = model.blocks[0].attn.n_head; hd = model.blocks[0].attn.head_dim
    sums = torch.zeros(L, H, hd, device=DEVICE); total = 0.0; ctx = {}
    def make(l):
        def pre(mod, args):
            y = args[0]; yy = y.view(y.shape[0], y.shape[1], H, hd)
            sums[l] += (yy * ctx["m"][:, :, None, None]).sum((0, 1))
        return pre
    hs = [model.blocks[l].attn.proj.register_forward_pre_hook(make(l)) for l in range(L)]
    try:
        for tk, nr in zip(toks, nreals):
            ar = torch.arange(tk.shape[1], device=DEVICE)
            for i in range(0, tk.shape[0], bs):
                ctx["m"] = (ar[None, :] < nr[i:i + bs][:, None]).float(); total += ctx["m"].sum().item()
                model(tk[i:i + bs])
    finally:
        for h in hs: h.remove()
    return sums / total

@torch.no_grad()
def ce_per_tree(model, tokens, nreal, bs=128, pre_hook_layer=None, pre_hook=None):
    ar = torch.arange(tokens.shape[1] - 1, device=DEVICE); out = []
    for i in range(0, tokens.shape[0], bs):
        x = tokens[i:i + bs]; nr = nreal[i:i + bs]; h = None
        if pre_hook is not None: h = pre_hook_layer.register_forward_pre_hook(pre_hook)
        try: logits, _ = model(x)
        finally:
            if h is not None: h.remove()
        m = (ar[None, :] <= (nr - 2)[:, None]).float(); out.append(((_ce(logits, x) * m).sum(1) / m.sum(1)).cpu())
    return torch.cat(out).numpy()

def head_ablation(model, core, tok, nreal, langs):
    L = len(model.blocks); H = model.blocks[0].attn.n_head; hd = model.blocks[0].attn.head_dim
    mu = head_mean_outputs(model, [tok[lg] for lg in langs], [nreal[lg] for lg in langs])
    base_t = {lg: ce_per_tree(model, tok[lg], nreal[lg]) for lg in langs}; N = len(base_t[langs[0]])
    dtree = {lg: np.zeros((L, H, N)) for lg in langs}                                  # per-tree CE change under ablation (paired with clean)
    for l in range(L):
        for h in range(H):
            def pre(mod, args, l=l, h=h):
                y = args[0].clone(); y[..., h * hd:(h + 1) * hd] = mu[l, h].to(y.dtype); return (y,)
            for lg in langs:
                dtree[lg][l, h] = ce_per_tree(model, tok[lg], nreal[lg], pre_hook_layer=model.blocks[l].attn.proj, pre_hook=pre) - base_t[lg]
    return dict(base={lg: float(base_t[lg].mean()) for lg in langs}, dce={lg: dtree[lg].mean(-1) for lg in langs}, dtree=dtree)

def ablation_specificity(dtree, a, b, z_min=4.0, effect_min=0.02):
    """Paired over trees (same trees, different rendering): d_t = dCE_b,t - dCE_a,t per head. A head is 'order-specific' if |mean d| >= effect_min
    and |z| >= z_min. -> (mean_diff (L,H), z (L,H), specific bool (L,H))"""
    d = dtree[b] - dtree[a]; m = d.mean(-1); se = d.std(-1, ddof=1) / np.sqrt(d.shape[-1]); z = m / np.clip(se, 1e-9, None)
    return m, z, (np.abs(m) >= effect_min) & (np.abs(z) >= z_min)

def _rank(a): return np.argsort(np.argsort(a)).astype(float)

def run_E2(model, langs, core, tok, nreal, levels=LEVELS_E2, comparisons=(("L_A", "L_C"), ("L_A", "L_B"), ("L_A", "L_A2"))):
    prof = {(lg, l): offset_profiles(model, core, tok[lg], lg, l) for lg in langs for l in levels}
    dist = {(a, b, l): profile_distances(prof[(a, l)], prof[(b, l)]) for a, b in comparisons if a in langs and b in langs for l in levels}
    abl = head_ablation(model, core, tok, nreal, langs)
    return dict(prof_bos={k: v["bos"].mean(0) for k, v in prof.items()}, dist=dist, abl=abl, levels=list(levels))

def report_E2(out, tag="", top_frac=0.25, n_show=12):
    abl = out["abl"]; langs = list(abl["dce"]); mx = np.max([abl["dce"][lg] for lg in langs], axis=0)       # importance = worst-case language
    matter = mx >= np.quantile(mx, 1 - top_frac)
    print(f"\n[{tag}] HEAD ABLATION (language-pooled mean ablation), CE change per language; baseline CE: " + ", ".join(f"{lg} {abl['base'][lg]:.3f}" for lg in langs))
    order = np.dstack(np.unravel_index(np.argsort(-mx, axis=None), mx.shape))[0][:n_show]
    print(f"  top {n_show} heads by max dCE   (layer,head)  " + "  ".join(f"d{lg[-1]}" for lg in langs))
    for l, h in order: print(f"    L{l:<2d}H{h}   " + "  ".join(f"{abl['dce'][lg][l, h]:+.3f}" for lg in langs))
    print("\n  ORDER-SPECIFIC HEADS (paired over trees; |mean diff| >= 0.02 nats and |z| >= 4).  A-vs-B is the nearer-order comparison, A-vs-A2 (if present) the null.")
    n_all = mx.size
    for b in [lg for lg in langs if lg != "L_A"]:
        m, z, sp = ablation_specificity(abl["dtree"], "L_A", b)
        big = np.abs(m) >= 0.02
        print(f"   A vs {b}: {int(sp.sum())}/{n_all} heads specific  ({int(big.sum())} with |diff|>=0.02)  | " +
              "  ".join(f"L{l}H{h}:{m[l, h]:+.3f}" for l, h in sorted(zip(*np.nonzero(sp)), key=lambda x: -abs(m[x]))[:6]))
    # ---- offset profiles: only interpretable if the control passes
    print("\n  OFFSET-PROFILE COMPARISON (A vs other). 'differs' = cross EMD > floor.")
    has_a2 = ("L_A", "L_A2", out["levels"][0]) in out["dist"]
    if has_a2: print("   floor = 95th percentile of EMD(A half, A' half): same order, different tag (the right null).")
    else:
        ctrl = out["dist"].get(("L_A", "L_B", 6))
        frac = float((ctrl["cross"] > ctrl["floor95"]).mean()) if ctrl else float("nan")
        print(f"   no A' model yet: floor = within-language split floor only. CONTROL: level-6 A-vs-B (identical span content) flags {frac:.0%} of heads.")
        if frac > 0.25: print("   >>> CONTROL FAILS: the split floor only captures sampling noise, not position/context differences. Profile comparison below is NOT interpretable. <<<")
    print(f"  {'cmp':>10s} {'lvl':>3s} | {'#differ':>7s} {'differ|matter':>13s} {'differ|other':>12s}")
    for (a, b, l), d in sorted(out["dist"].items(), key=lambda kv: (kv[0][1], kv[0][2])):
        if b == "L_A2": continue
        if has_a2: floor = np.percentile(out["dist"][("L_A", "L_A2", l)]["cross_all"], 95, axis=0)
        else: floor = d["floor95"]
        differs = d["cross"] > floor
        fm = differs[matter].mean() if matter.any() else float("nan"); fo = differs[~matter].mean() if (~matter).any() else float("nan")
        print(f"  {a[-1]}-vs-{b[-1]:>6s} {l:>3d} | {int(differs.sum()):>3d}/{differs.size:<3d} {fm:13.2f} {fo:12.2f}")
    print("  Caveat: attention at span ends is a thin slice of what a head does; identical offset profiles do not imply identical content.")

E2_OUT = {}
if RUN_E2:
    for name in EVAL_MODELS:
        if not os.path.exists(CKPTS[name]["path"]): print(f"[skip] {name}"); continue
        print(f"=== E2 on {name} ==="); model, langs = load_model(CKPTS[name]["path"], CKPTS[name]["n_head"])
        E2_OUT[name] = run_E2(model, langs, core, tok, nreal); report_E2(E2_OUT[name], tag=name)
        del model; torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
    os.makedirs("./exp7_out", exist_ok=True); pickle.dump(E2_OUT, open("./exp7_out/e2_results.pkl", "wb"))
else: print("RUN_E2 is False")


=== E2 on bp_h4_d96_s0 ===
  ./exp6_bigpool_h4/shared_d96_h4_seed0.pt: vocab=7 d=96 layers=12 heads=4 step=90000 tail-1000 loss=0.7973 langs=['L_A', 'L_B', 'L_C']

[bp_h4_d96_s0] HEAD ABLATION (language-pooled mean ablation), CE change per language; baseline CE: L_A 0.772, L_B 0.778, L_C 0.782
  top 12 heads by max dCE   (layer,head)  dA  dB  dC
    L3 H1   +0.142  +0.157  +0.144
    L7 H1   +0.004  +0.017  +0.133
    L2 H3   +0.130  +0.102  +0.111
    L0 H2   +0.115  +0.109  +0.035
    L11H1   +0.011  +0.020  +0.008
    L1 H2   +0.017  +0.012  +0.004
    L4 H2   +0.015  +0.004  +0.001
    L9 H0   +0.001  +0.003  +0.015
    L9 H2   +0.004  +0.004  +0.012
    L5 H2   +0.011  +0.012  +0.000
    L8 H3   +0.011  +0.002  +0.001
    L11H2   +0.008  +0.008  +0.011

  ORDER-SPECIFIC HEADS (paired over trees; |mean diff| >= 0.02 nats and |z| >= 4).  A-vs-B is the nearer-order comparison, A-vs-A2 (if present) the null.
   A vs L_B: 1/48 heads specific  (1 with |diff|>=0.02)  | L2H3:-0.027
   A v

## 5. E4 — A' calibration run (training, background)

In [7]:
# =============================== E4: A' calibration run (duplicate anchor) ===============================
# A' = same linearization as A, different BOS tag. It only means something if the model is TRAINED with it, so this is a new
# 4-language model (vocab 8), not an extra eval on the old checkpoints. Mix keeps A+A' = 70% so B and C keep exactly 15% each;
# consequence: A and A' each see HALF the sequences that A saw in the 3-language runs. Compare A vs A' inside this one model;
# comparing this model's A to the old models' A carries that mild confound. Everything else = nb6 big-pool recipe:
# d=96, n_head=4, 12 layers, 500K-tree pool, batch 32, lr 3e-3 AdamW, 90K steps fixed (no convergence flag), bf16 autocast.
E4 = dict(root="./exp7_e4", big_n=500_000, chunk=50_000, d=96, n_head=4, seed=0, data_seed=1, steps=90_000,
          mix={"L_A": 0.35, "L_A2": 0.35, "L_B": 0.15, "L_C": 0.15})
assert abs(E4["mix"]["L_A"] + E4["mix"]["L_A2"] - 0.70) < 1e-9 and E4["mix"]["L_B"] == 0.15 and E4["mix"]["L_C"] == 0.15

def make_mix_stream(rules, n_trees, seed, seq_len, mix):
    rng = random.Random(seed); names = list(mix); w = [mix[n] for n in names]; stream = []
    for _ in range(n_trees):
        lang = rng.choices(names, weights=w)[0]; tree = sample_tree(rules, rng)
        stream.append(BOS_ALL[lang]); stream.extend(linearize(tree, LANGS[lang]))
    s = torch.tensor(stream, dtype=torch.long); n = len(s) // (seq_len + 1)
    return s[: n * (seq_len + 1)].view(n, seq_len + 1)

def gen_pool(make_fn, n_total, chunk, seed0):
    parts, t0 = [], time.time()
    for i, start in enumerate(range(0, n_total, chunk)):
        n = min(chunk, n_total - start); parts.append(make_fn(n, seed0 * 1000 + i).cpu())
        if i == 0: el = time.time() - t0; print(f"    first chunk ({n} trees): {el:.0f}s -> est. total {el * n_total / n / 60:.1f} min", flush=True)
    return torch.cat(parts)

E4_DATA = os.path.join(E4["root"], f"data_shared4_seed{E4['data_seed']}.pt")
E4_CKPT = CKPTS["e4_a2_d96_s0"]["path"]
if RUN_E4_DATAGEN and not os.path.exists(E4_DATA):
    os.makedirs(E4["root"], exist_ok=True); rules_e4, _ = build_grammar(GRAMMAR_SEED)
    pool = gen_pool(lambda n, sd: make_mix_stream(rules_e4, n, sd, SEQ_LEN, E4["mix"]), E4["big_n"], E4["chunk"], E4["data_seed"])
    torch.save(pool, E4_DATA); print(f"saved {tuple(pool.shape)} -> {E4_DATA}")
    # the pool must contain all four tags in the intended proportions
    first = pool[:, 0]; print({n: round(float((first == b).float().mean()), 3) for n, b in BOS_ALL.items()}, "(row-start tags only; rows are packed streams)")

# worker: verbatim copy of nb6's train_worker.py (written only if absent, so an existing worker is reused untouched)
WORKER_SRC = r'''import argparse, os, time
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- model: verbatim copy of the notebook's model cell ----
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert (n_embd // n_head) % 2 == 0, "head_dim must be EVEN for rotary embeddings"
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

# ---- training: same recipe as train_fixed_budget ----
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--d", type=int, required=True)
    ap.add_argument("--seed", type=int, required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--vocab", type=int, required=True)
    ap.add_argument("--seq_len", type=int, required=True)
    ap.add_argument("--steps", type=int, default=90000)
    ap.add_argument("--n_head", type=int, default=4)
    ap.add_argument("--n_layer", type=int, default=12)
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--lr", type=float, default=3e-3)
    ap.add_argument("--compile", type=int, default=0)
    a = ap.parse_args()

    torch.set_num_threads(1)                     # avoid CPU oversubscription with several workers
    device = torch.device(a.device)
    torch.manual_seed(a.seed)
    data = torch.load(a.data).to(device)
    model = TinyRotaryGPT(a.vocab, n_embd=a.d, n_layer=a.n_layer, n_head=a.n_head,
                          max_seq_len=a.seq_len + 1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, fused=True)
    fwd = torch.compile(model) if a.compile else model
    n = data.shape[0]
    buf = torch.zeros(a.steps, device=device)
    t0 = time.time()
    for step in range(a.steps):
        idx = torch.randint(0, n, (a.batch,), device=device)
        batch = data[idx]
        x, y = batch[:, :-1], batch[:, 1:]
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            _, loss = fwd(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
        buf[step] = loss.detach().float()
        if (step + 1) % 5000 == 0:
            print(f"step {step+1:>7d}: mean loss over last 5000 = {buf[step-4999:step+1].mean().item():.4f}"
                  f"  ({(step + 1) / (time.time() - t0):.0f} it/s)", flush=True)
        if (step + 1) % 10000 == 0 or step == a.steps - 1:
            tmp = a.ckpt + ".tmp"
            torch.save({"model_state": model.state_dict(), "opt_state": opt.state_dict(),
                        "step": step + 1, "losses": buf[:step + 1].tolist()}, tmp)
            os.replace(tmp, a.ckpt)

if __name__ == "__main__":
    main()'''
if not os.path.exists("train_worker.py"):
    open("train_worker.py", "w").write(WORKER_SRC); print("wrote train_worker.py (copy of nb6's worker)")

def ckpt_step(path):
    if not os.path.exists(path): return -1
    return torch.load(path, map_location="cpu", weights_only=False)["step"]

E4_PROC = None
if RUN_E4_LAUNCH:
    assert os.path.exists(E4_DATA), "run with RUN_E4_DATAGEN=True first"
    if ckpt_step(E4_CKPT) == E4["steps"]: print("E4 checkpoint already complete:", E4_CKPT)
    else:
        os.makedirs(E4["root"], exist_ok=True); log = open(os.path.join(E4["root"], "log_e4_a2_d96_s0.txt"), "w")
        cmd = [sys.executable, "train_worker.py", "--d", str(E4["d"]), "--seed", str(E4["seed"]), "--data", E4_DATA, "--ckpt", E4_CKPT,
               "--vocab", str(len(BOS_ALL) + TERM_SIZE), "--seq_len", str(SEQ_LEN), "--steps", str(E4["steps"]),
               "--n_head", str(E4["n_head"]), "--device", str(DEVICE)]
        E4_PROC = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
        print("launched E4 in the background, pid", E4_PROC.pid, "| log:", log.name, "| ckpt:", E4_CKPT)
        print("When it finishes: add 'e4_a2_d96_s0' to EVAL_MODELS and re-run the E1 cell. E1 then also reports L_A2->L_A (the A' ceiling).")
else: print("RUN_E4_LAUNCH is False -- nothing launched.")


    first chunk (50000 trees): 6s -> est. total 0.9 min
saved (464206, 257) -> ./exp7_e4/data_shared4_seed1.pt
{'L_A': 0.001, 'L_B': 0.001, 'L_C': 0.001, 'L_A2': 0.002} (row-start tags only; rows are packed streams)
launched E4 in the background, pid 627307 | log: ./exp7_e4/log_e4_a2_d96_s0.txt | ckpt: ./exp7_e4/shared4_d96_h4_seed0.pt
When it finishes: add 'e4_a2_d96_s0' to EVAL_MODELS and re-run the E1 cell. E1 then also reports L_A2->L_A (the A' ceiling).


## 6. Pre-registration and verdict (refuses to give a verdict until thresholds are frozen)

In [8]:
# =============================== PRE-REGISTRATION (E1 v2, E2) ===============================
# HONEST STATUS: the E1 instrument was REVISED after a first look at A->C numbers (v1 had no dynamic range at levels 4-5). A conclusion from
# v2 is therefore not strictly pre-registered. What can still be kept clean: (a) thresholds are set ONLY from control rows (identity,
# noise/junk, within-order, level-6 A/B, and A'->A once E4 exists), never from A<->C rows; (b) confirmation comes from seeds/models
# NOT used to set thresholds (e.g. set on seed 0, test on seeds 7 and 13; or on E4's model).
# The layer used for a verdict is fixed by RULE: per recipient, the layer with the largest 'reads' strength (swap - junk, averaged over
# levels 4-5) -- a property of the recipient's own within-order data, not of any cross-order number.
PREREG_TABLE = """
| Exp | Surprising reading                                                                                 | Boring reading                                                     |
| E1b | S_code(A<->C) near floor at levels 4-5 WITH validity (model reads the code), A'->A near ceiling,   | S_code(A<->C) not near floor, or no validity ('reads?' = NO)        |
|     | level-6 A/B near ceiling                                                                           |                                                                    |
| E1  | S_nt_m(A<->C) near floor at 4-5 with range z >> 3 (secondary, whole-residual)                      | no range, or not near floor                                        |
| E2  | ablation: many heads order-specific vs the A-vs-A' null; profiles (only if control passes) differ  | specificity within the A-vs-A' null; or profile control fails      |
"""
THRESH = dict(e1_near_floor_S=None, e1_near_ceiling_S=None, e1_min_range=None)
THRESH_FILE = "./exp7_out/prereg_frozen.json"

def freeze_thresholds():
    missing = [k for k, v in THRESH.items() if v is None]
    assert not missing, f"fill these first: {missing}"
    os.makedirs("./exp7_out", exist_ok=True); json.dump(dict(THRESH, frozen_at=time.strftime("%Y-%m-%d %H:%M:%S")), open(THRESH_FILE, "w"), indent=2)
    print("frozen ->", THRESH_FILE)

def _frozen():
    if not os.path.exists(THRESH_FILE): return None
    j = json.load(open(THRESH_FILE)); return j if all(j.get(k) == v for k, v in THRESH.items()) else None

def verdict_E1(out_by_model):
    print(PREREG_TABLE)
    fz = _frozen()
    if fz is None:
        print("UNFROZEN thresholds -> no verdict. Read the raw tables; set THRESH from control rows only; freeze_thresholds(); re-run."); return
    for name, out in out_by_model.items():
        res = out["res"]; print(f"--- {name} ---")
        def layer_for(R):
            st = {k: np.mean([(res[(R, l, k, "cb:swap")]["w"] - res[(R, l, k, "cb:junk")]["w"]).mean() for l in (4, 5)]) for k in out["layers"]}
            return max(st, key=st.get)
        def code(R, D, l, k, ms=fz["e1_min_range"]): return index_code(res, R, l, k, D, min_range=ms)
        rows = {}
        for R, D in (("L_C", "L_A"), ("L_A", "L_C")):
            if (D, R) not in out["pairs"]: continue
            k = layer_for(R); rows[(R, D)] = [code(R, D, l, k) for l in (4, 5)]
            print(f"  recipient {R} <- {D} at layer {k} (chosen by rule): " + "  ".join(f"lvl{l}: S_code={r['S']:.2f} ok={r['ok']} reads={r['reads']}" for l, r in zip((4, 5), rows[(R, D)])))
        l6 = code("L_B", "L_A", 6, layer_for("L_B")) if ("L_A", "L_B") in out["pairs"] else None
        ceil = code("L_A", "L_A2", 4, layer_for("L_A")) if ("L_A2", "L_A") in out["pairs"] else None
        print(f"  level-6 A->B control: {None if l6 is None else round(l6['S'], 2)}     A'->A ceiling: {None if ceil is None else round(ceil['S'], 2)}")
        allr = [r for v in rows.values() for r in v]
        if not allr or not all(r["ok"] for r in allr): print("  READING: NO DYNAMIC RANGE / model does not read the code at the rule-chosen layer -> E1b cannot decide."); continue
        near_floor = all(r["S"] <= fz["e1_near_floor_S"] for r in allr)
        ceil_ok = ceil is None or (ceil["ok"] and ceil["S"] >= fz["e1_near_ceiling_S"]); l6_ok = l6 is None or (l6["ok"] and l6["S"] >= fz["e1_near_ceiling_S"])
        print("  READING:", ("SURPRISING-side: non-sharing of the NT code at levels 4-5, controls OK" if near_floor and ceil_ok and l6_ok else
                             "INSTRUMENT SUSPECT: controls fail" if near_floor else "BORING-side: A<->C code not near floor"))
        if ceil is None: print("  note: no A' ceiling yet (E4 untrained) -> provisional")

if E1_OUT: verdict_E1(E1_OUT)



| Exp | Surprising reading                                                                                 | Boring reading                                                     |
| E1b | S_code(A<->C) near floor at levels 4-5 WITH validity (model reads the code), A'->A near ceiling,   | S_code(A<->C) not near floor, or no validity ('reads?' = NO)        |
|     | level-6 A/B near ceiling                                                                           |                                                                    |
| E1  | S_nt_m(A<->C) near floor at 4-5 with range z >> 3 (secondary, whole-residual)                      | no range, or not near floor                                        |
| E2  | ablation: many heads order-specific vs the A-vs-A' null; profiles (only if control passes) differ  | specificity within the A-vs-A' null; or profile control fails      |

UNFROZEN thresholds -> no verdict. Read the raw tables; set THRESH from control rows only; freeze_threshold

## Not built here
- **E3 NoPE swap** — needs retraining at d=96, same budget/seeds; do after E1 is clean.
- **E5 bidirectional control** — keep the objective fixed and change only the mask (same cloze/masked-span objective, causal vs full mask). Remaining confounds to settle before building: (i) under a full mask, A and C are exact mirror images, so sharing there can be achieved by sign-flip of relative offsets and says nothing about B; (ii) a full-mask cloze model can solve the task with local neighbours instead of parse-level computation, so a short mask span rate matters.
- **E6 sharing push** — needs aligned NT labels at span ends (available via `SE`) and the per-order-head control; start only once E1 and E4 give a usable ceiling.
- Rho from Pedra Bento (NeurIPS 2026): not implemented; I have not read the paper and will not reconstruct the invariant from memory.


In [1]:
!setsid nohup python train_worker.py --d 96 --seed 0 --data ./exp7_e4/data_shared4_seed1.pt --ckpt ./exp7_e4/shared4_d96_h4_seed0.pt --vocab 8 --seq_len 256 --steps 90000 --n_head 4 --device cuda:1 > exp7_e4/log_e4_train.txt 2>&1 &

OSError: Background processes not supported.

In [2]:
import subprocess, os
assert os.path.exists("train_worker.py"), "train_worker.py not in this folder"
log = open("exp7_e4/log_e4_train.txt", "w")
p = subprocess.Popen(
    ["python", "train_worker.py", "--d", "96", "--seed", "0",
     "--data", "./exp7_e4/data_shared4_seed1.pt",
     "--ckpt", "./exp7_e4/shared4_d96_h4_seed0.pt",
     "--vocab", "8", "--seq_len", "256", "--steps", "90000",
     "--n_head", "4", "--device", "cuda:1"],
    stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("pid", p.pid)

pid 2658740


In [12]:
# =====================================================================================================
# Exp 7 instrument fixes. Paste each "# %%" block as its own cell AFTER cell 9 of
# Experiment7_v2_SerializationSharing.ipynb. It reuses that notebook's names:
#   DEVICE, LANGS, NT_SIZE, LEVELS, core, tok, nreal, load_model, hidden_states, build_code_tables,
#   select_recipients, draw_donors_matched, _seed, _boot, _se, CKPTS, EVAL_MODELS
# Nothing here retrains anything. All eval is fp32 (TF32 already off in cell 2).
# =====================================================================================================

# %% ---------------------------------- CELL 1: shared helpers ----------------------------------
import random, math, numpy as np, torch, torch.nn.functional as F
from collections import defaultdict, Counter

def assert_e4_done(path, step=90_000):
    """Hard gate: refuse to evaluate the A' model before training finished (the worker has no resume)."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    assert ck["step"] >= step, f"E4 not finished: checkpoint step {ck['step']} < {step}. Do not evaluate yet."
    print(f"E4 checkpoint at step {ck['step']} -> OK to evaluate")

@torch.no_grad()
def next_logits(model, tokens, rt, rp, k=None, vecs=None, bs=128):
    """Logits at position rp (the prediction of token rp+1), optionally with block-k output at rp replaced by vecs.
    W=1 readout: this is the next-token distribution right after the span end."""
    out = []
    for i in range(0, rt.shape[0], bs):
        t = rt[i:i + bs]; p = rp[i:i + bs]; x = tokens[t]; ab = torch.arange(x.shape[0], device=DEVICE); hk = None
        if vecs is not None:
            v = vecs[i:i + bs]
            def hook(m, inp, o, v=v, p=p, ab=ab):
                o = o.clone(); o[ab, p] = v.to(o.dtype); return o
            hk = model.blocks[k].register_forward_hook(hook)
        try: logits, _ = model(x)
        finally:
            if hk is not None: hk.remove()
        out.append(logits[ab, p].float())
    return torch.cat(out)

def kl_and_flip(clean, patched):
    """KL(p_clean || p_patched) in nats per recipient, and whether the argmax next token changed."""
    lc = F.log_softmax(clean, -1); lp = F.log_softmax(patched, -1)
    return (lc.exp() * (lc - lp)).sum(-1), (clean.argmax(-1) != patched.argmax(-1)).float()

def token_out(X, ks):
    """Remove per-(current, previous)-token means so the token-identity confound at span ends is gone."""
    uk, inv = torch.unique(ks, return_inverse=True)
    sums = torch.zeros(len(uk), X.shape[1], device=X.device, dtype=X.dtype).index_add_(0, inv, X)
    cnt = torch.bincount(inv, minlength=len(uk)).clamp(min=1).unsqueeze(1).to(X.dtype)
    return X - (sums / cnt)[inv]

def span_end_data(core, tok, H, lang, level, layer, held):
    """Span-end residuals for one language/level/layer. held=False -> FIT trees (t%3 != 0); held=True -> EVAL trees (t%3 == 0)."""
    items = [(k, v) for k, v in core["SE"][lang][level].items() if ((k[0] % 3) == 0) == held]
    ts = torch.tensor([k[0] for k, _ in items], device=DEVICE); ps = torch.tensor([v[1] for _, v in items], device=DEVICE)
    nts = torch.tensor([v[2] for _, v in items], device=DEVICE)
    key = tok[lang][ts, ps] * 8 + tok[lang][ts, ps - 1]
    return H[lang][layer][ts, ps], key, nts

def fit_lda_subspace(X, key, y, kmax=3, ridge=1e-3):
    """LDA on token-out span-end residuals. Returns an orthonormal basis Q (d, kmax), columns ordered by discriminability,
    so Q[:, :k] is the rank-k subspace. NT_SIZE=4 classes -> at most 3 discriminant directions exist."""
    Xt = token_out(X.double(), key); Xt = Xt - Xt.mean(0); y = y.to(Xt.device); d = Xt.shape[1]
    assert all((y == c).any() for c in range(NT_SIZE)), "a class is missing from the fit set"
    mu = torch.stack([Xt[y == c].mean(0) for c in range(NT_SIZE)])
    Xw = Xt - mu[y]; Sw = (Xw.T @ Xw) / len(Xt); Sw = Sw + ridge * Sw.trace() / d * torch.eye(d, device=Xt.device, dtype=Xt.dtype)
    pi = torch.stack([(y == c).double().mean() for c in range(NT_SIZE)])
    Sb = (mu.T * pi) @ mu
    ev, V = torch.linalg.eigh(Sw); Wm = V @ torch.diag(ev.clamp(min=1e-10).rsqrt()) @ V.T
    e2, V2 = torch.linalg.eigh(Wm @ Sb @ Wm); order = e2.argsort(descending=True)[:kmax]
    Q, _ = torch.linalg.qr(Wm @ V2[:, order])
    return Q.float(), e2[order].float()

def interchange(h, hd, Q):
    """h' = h - P h + P h_donor, P = Q Q^T (orthogonal projector onto the fitted subspace)."""
    return h + (hd - h) @ Q @ Q.T

def draw_matched_heldout(core, level, recs, rep, seed=0):
    """Same as draw_donors_matched (same ancestor chain + child index; same / different symbol) but donors come ONLY from
    eval trees (t%3==0), so the fitted subspace never saw a recipient or a donor."""
    rng = random.Random(_seed(seed, level, rep, "matched_heldout")); kept, same, diff = [], [], []
    for (t, inst, nt) in recs:
        chain, cidx = core["META"][level][(t, inst)]; by = core["BY"][(level, chain, cidx)]
        s_c = [x for x in by.get(nt, []) if x[0] != t and x[0] % 3 == 0]
        d_c = [x for n2, lst in by.items() if n2 != nt for x in lst if x[0] != t and x[0] % 3 == 0]
        if not s_c or not d_c: continue
        kept.append((t, inst, nt)); same.append(s_c[rng.randrange(len(s_c))]); diff.append(d_c[rng.randrange(len(d_c))])
    return kept, {"same_nt_m": same, "diff_nt_m": diff}


# %% ---------------------------------- CELL 2: E1b scale check (minutes, no retraining) ----------------------------------
# Question: are the E1b damages <= 0.02 nats because the code perturbation is tiny relative to the residual (scale),
# or because the class-mean direction is not what the model reads (probe class)?
#  - prints ||code||/||h||
#  - scales the WHOLE intervention:  h + s*(new_code - own_code), s in (1,4,16)  for within / swap / junk (junk = random, matched norm)
#  - positive control: damage of a FULL ancestor-matched wrong-symbol residual patch (diff_nt_m). 'recov' = swap_s / full.
#    recov near 0 even at s=16  -> the class-mean code is the wrong probe.   recov grows with s -> it was scale.
# Readout is KL at W=1, not 8-token CE.

def run_E1b_scale(model, langs, core, tok, nreal, R="L_A", levels=(4, 5), layers=(2, 4, 5, 6, 7), scales=(1, 4, 16), n_rep=4):
    H = {lg: hidden_states(model, tok[lg], layers) for lg in langs}
    tables = build_code_tables(core, H, tok, [R], levels, layers); SE = core["SE"]; acc = {}; ratio = defaultdict(list)
    for l in levels:
        for rep in range(n_rep):
            recs0 = select_recipients(core, R, l, rep); recs, dn = draw_donors_matched(core, l, recs0, rep)
            rt = torch.tensor([r[0] for r in recs], device=DEVICE); rp = torch.tensor([SE[R][l][(r[0], r[1])][1] for r in recs], device=DEVICE)
            nts = torch.tensor([r[2] for r in recs], device=DEVICE); n = rt.shape[0]; clean = next_logits(model, tok[R], rt, rp)
            dl = dn["diff_nt_m"]; dt = torch.tensor([x[0] for x in dl], device=DEVICE); dp = torch.tensor([SE[R][l][x][1] for x in dl], device=DEVICE)
            rem_s, rep_s = (rt + 1) % 3, (rt + 2) % 3
            for k in layers:
                TR = tables[(R, l, k)]; h = H[R][k][rt, rp]; own = TR[rem_s, nts]; code_in = TR[rep_s, nts]
                g = torch.Generator(device=DEVICE); g.manual_seed(_seed("scale", l, rep, k))
                other = (nts + torch.randint(1, NT_SIZE, (n,), generator=g, device=DEVICE)) % NT_SIZE; code_oth = TR[rep_s, other]
                junk = torch.randn(n, h.shape[1], generator=g, device=DEVICE); junk = junk / junk.norm(dim=1, keepdim=True) * code_oth.norm(dim=1, keepdim=True)
                ratio[(l, k)].append((code_oth.norm(dim=1) / h.norm(dim=1)).cpu())
                conds = {"ident": h, "full:diff": H[R][k][dt, dp]}
                for s in scales:
                    conds[f"s{s}:within"] = h + s * (code_in - own); conds[f"s{s}:swap"] = h + s * (code_oth - own); conds[f"s{s}:junk"] = h + s * (junk - own)
                for name, v in conds.items():
                    kl, _ = kl_and_flip(clean, next_logits(model, tok[R], rt, rp, k=k, vecs=v)); acc.setdefault((l, k, name), []).append(kl.cpu())
    res = {key: torch.cat(v).numpy() for key, v in acc.items()}
    assert max(abs(res[(l, k, "ident")].mean()) for l in levels for k in layers) < 1e-6, "identity patch changed the logits: plumbing bug"
    for l in levels:
        print(f"\n[E1b scale | recipient {R} | level {l}]  KL(clean||patched) at W=1, nats.   z = (swap-junk)/SE.   recov = swap / full wrong-symbol patch")
        for k in layers:
            full = res[(l, k, "full:diff")].mean(); line = f"k={k} |code|/|h|={torch.cat(ratio[(l, k)]).mean():.3f} full={full:.4f} |"
            for s in scales:
                w, sw, jk = res[(l, k, f"s{s}:within")], res[(l, k, f"s{s}:swap")], res[(l, k, f"s{s}:junk")]; d = sw - jk
                line += f" x{s}: within {w.mean():.4f} swap {sw.mean():.4f} junk {jk.mean():.4f} z {d.mean() / _se(d):5.1f} recov {sw.mean() / max(full, 1e-9):5.2f} |"
            print(line)
    return res


# %% ---------------------------------- CELL 3: fitted-subspace interchange + KL readout + transfer ----------------------------------
# For fit language F in fit_langs, recipient R, level l, layer k, rank r:
#   S = LDA subspace fit on F's span-end residuals (token-out) from FIT trees; all recipients/donors come from HELD-OUT trees.
#   C0 = KL of interchange with a SAME-symbol, ancestor-matched donor in R        (ceiling: context/estimation noise)
#   Fm = KL of interchange with a DIFFERENT-symbol, ancestor-matched donor in R   (does R read S?)
#   RND = same as Fm with a random rank-r subspace (floor for 'S is special'); 'reads' needs Fm > RND.
#   XN(D) = KL with a SAME-symbol donor taken from language D                     (the sharing test)
#   S_nt = 1 - (XN - C0)/(Fm - C0)       1 = donor language's S-content is as harmless as R's own; 0 = as harmful as a wrong symbol.
# Transfer asymmetry check: run with fit on A and fit on C; if the picture depends on which one you fit on, report it that way.
# A' ceiling: when the 4-language E4 model is loaded, L_A2 appears as a donor language automatically.

def run_E1c(model, langs, core, tok, nreal, recips=("L_A", "L_C"), fit_langs=("L_A", "L_C"), levels=(4, 5),
            layers=(2, 4, 6), ranks=(1, 2, 3), n_rep=4, n_rand=4, min_recs=30):
    H = {lg: hidden_states(model, tok[lg], layers) for lg in langs}; SE = core["SE"]; acc = {}; Q = {}
    for Fl in fit_langs:
        for l in levels:
            for k in layers:
                X, key, y = span_end_data(core, tok, H, Fl, l, k, held=False); Q[(Fl, l, k)] = fit_lda_subspace(X, key, y, kmax=max(ranks))[0]
    for l in levels:
        for rep in range(n_rep):
            for R in recips:
                recs0 = [r for r in select_recipients(core, R, l, rep) if r[0] % 3 == 0]; recs, dn = draw_matched_heldout(core, l, recs0, rep)
                if len(recs) < min_recs: print(f"  [skip] {R} level {l} rep {rep}: only {len(recs)} recipients"); continue
                rt = torch.tensor([r[0] for r in recs], device=DEVICE); rp = torch.tensor([SE[R][l][(r[0], r[1])][1] for r in recs], device=DEVICE)
                clean = next_logits(model, tok[R], rt, rp)
                def dvec(D, mode, k):
                    dl = dn[mode]; dt = torch.tensor([x[0] for x in dl], device=DEVICE); dp = torch.tensor([SE[D][l][x][1] for x in dl], device=DEVICE)
                    return H[D][k][dt, dp]
                for k in layers:
                    h = H[R][k][rt, rp]
                    for Fl in fit_langs:
                        for rk in ranks:
                            Qk = Q[(Fl, l, k)][:, :rk]; conds = {"ident": h}
                            for mode in ("same_nt_m", "diff_nt_m"):
                                for D in langs: conds[f"{D}:{mode}"] = interchange(h, dvec(D, mode, k), Qk)
                            for j in range(n_rand):
                                g = torch.Generator(device=DEVICE); g.manual_seed(_seed(l, rep, R, k, rk, j))
                                Qr = torch.linalg.qr(torch.randn(h.shape[1], rk, generator=g, device=DEVICE))[0]
                                conds[f"rand{j}"] = interchange(h, dvec(R, "diff_nt_m", k), Qr)
                            for name, v in conds.items():
                                kl, fl = kl_and_flip(clean, next_logits(model, tok[R], rt, rp, k=k, vecs=v))
                                acc.setdefault((Fl, R, l, k, rk, name), []).append((kl.cpu(), fl.cpu()))
    res = {key: (torch.cat([a for a, _ in v]).numpy(), torch.cat([b for _, b in v]).numpy()) for key, v in acc.items()}
    worst = max(abs(v[0].mean()) for key, v in res.items() if key[-1] == "ident"); print(f"  identity patches: max |KL| = {worst:.2e}"); assert worst < 1e-6
    return dict(res=res, langs=list(langs), recips=list(recips), fit_langs=list(fit_langs), levels=list(levels), layers=list(layers), ranks=list(ranks), n_rand=n_rand)

MIN_RANGE_KL = 1e-3

def report_E1c(out, tag=""):
    res = out["res"]
    for Fl in out["fit_langs"]:
        for R in out["recips"]:
            for l in out["levels"]:
                print(f"\n[{tag}] fit on {Fl} | recipient {R} | level {l} | KL at W=1 (nats) | flip = argmax-changed rate")
                print(f"{'k':>2s} {'r':>1s} | {'C0':>6s} {'Fm':>6s} {'RND':>6s} {'reads':>5s} {'flip':>5s} |  donor:  XN  S_nt [5%,95%]")
                for k in out["layers"]:
                    for rk in out["ranks"]:
                        g = lambda n: res[(Fl, R, l, k, rk, n)][0]
                        if (Fl, R, l, k, rk, f"{R}:same_nt_m") not in res: continue
                        C0, Fm = g(f"{R}:same_nt_m"), g(f"{R}:diff_nt_m"); RND = np.mean([g(f"rand{j}") for j in range(out["n_rand"])], axis=0)
                        reads = (Fm - RND).mean() > 3 * _se(Fm - RND); rng = Fm - C0; ok = reads and rng.mean() > max(MIN_RANGE_KL, 3 * _se(rng))
                        flip = res[(Fl, R, l, k, rk, f"{R}:diff_nt_m")][1].mean(); line = f"{k:>2d} {rk:>1d} | {C0.mean():6.4f} {Fm.mean():6.4f} {RND.mean():6.4f} {('yes' if reads else 'NO'):>5s} {flip:5.2f} |"
                        for D in out["langs"]:
                            if D == R: continue
                            XN = g(f"{D}:same_nt_m")
                            if ok:
                                S = 1 - (XN.mean() - C0.mean()) / rng.mean()
                                ci = _boot(lambda m: np.where(m["F"] - m["C"] > 0, 1 - (m["X"] - m["C"]) / (m["F"] - m["C"]), np.nan), dict(F=Fm, C=C0, X=XN))
                                line += f" {D[2:]}: {XN.mean():6.4f} {S:5.2f}[{ci[0]:5.2f},{ci[1]:5.2f}]"
                            else: line += f" {D[2:]}: {XN.mean():6.4f}   nan (no range / not read)"
                        print(line)
    print("\nS_nt=1: donor language's content of the fitted subspace is as harmless as R's own; 0: as harmful as a wrong symbol. nan = Fm not above RND, or no range.")
    print("A2 rows (only in the E4 model) are the tag-only ceiling: order effect = (S_nt for A<-A2) minus (S_nt for A<-B or A<-C).")


# %% ---------------------------------- CELL 4: does the readout depend on the finished child's symbol at all? ----------------------------------
# Held-out cross-entropy of the NEXT TOKEN after a level-l span end, given (ancestor chain, child index) vs given (chain, child index, symbol).
# Gap = what the finished child's symbol adds to predicting the next token. Gap ~ 0 means E1b damages <= 0.02 are expected, not weak.
# Plug-in counts on half the trees, scored on the other half (no optimistic bias from the finer partition). Needs a bigger tree sample
# than the 600 eval trees, so it builds its own.

def readout_dependence(n_trees=6000, seed=4242, levels=(4, 5, 6), alpha=0.5, langs=("L_A", "L_C")):
    c = build_eval_core(build_grammar(GRAMMAR_SEED)[0], n_trees, seed)
    for lang in langs:
        for l in levels:
            rows = []
            for (t, inst), (f, la, idx, cnt) in c["SE"][lang][l].items():
                if la + 1 > c["nreal"][lang][t] - 1: continue                 # span end is the last real token: no next token
                chain, cidx = c["META"][l][(t, inst)]; rows.append((t, (chain, cidx), idx, int(c["tokens"][lang][t, la + 1])))
            def ce(ctxfn, fit, ev):
                cnt = defaultdict(Counter); V = 8
                for r in fit: cnt[ctxfn(r)][r[3]] += 1
                tot = 0.0
                for r in ev:
                    cc = cnt[ctxfn(r)]; tot -= math.log((cc[r[3]] + alpha) / (sum(cc.values()) + alpha * V))
                return tot / len(ev)
            fit = [r for r in rows if r[0] % 2 == 0]; ev = [r for r in rows if r[0] % 2 == 1]
            a = ce(lambda r: r[1], fit, ev); b = ce(lambda r: (r[1], r[2]), fit, ev)
            print(f"[{lang} level {l}] n={len(rows):6d} | CE(next | chain,cidx)={a:.4f}  CE(next | chain,cidx,symbol)={b:.4f}  gain from symbol={a - b:.4f} nats")


# %% ---------------------------------- CELL 5: driver ----------------------------------
# Run order: (1) readout_dependence, (2) scale check on seed 0, (3) fitted-subspace run on all 3 seeds,
# (4) the same on the E4 model ONLY after assert_e4_done passes. Keep decision rules written down BEFORE step (4).
RUN_READOUT_CHECK = True; RUN_SCALE = True; RUN_FIT = True; RUN_E4_EVAL = False       # flip RUN_E4_EVAL only when the log shows step 90000
if RUN_READOUT_CHECK: readout_dependence()
E1B, E1C = {}, {}
for name in EVAL_MODELS:
    if not os.path.exists(CKPTS[name]["path"]): print(f"[skip] {name}"); continue
    model, langs = load_model(CKPTS[name]["path"], CKPTS[name]["n_head"])
    if RUN_SCALE and name == EVAL_MODELS[0]: E1B[name] = run_E1b_scale(model, langs, core, tok, nreal)
    if RUN_FIT: E1C[name] = run_E1c(model, langs, core, tok, nreal); report_E1c(E1C[name], tag=name)
    del model; torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
if RUN_E4_EVAL:
    p = CKPTS["e4_a2_d96_s0"]["path"]; assert_e4_done(p); model, langs = load_model(p, CKPTS["e4_a2_d96_s0"]["n_head"])
    E1C["e4"] = run_E1c(model, langs, core, tok, nreal, recips=("L_A", "L_A2", "L_C")); report_E1c(E1C["e4"], tag="e4")
    del model
pickle.dump(dict(E1B=E1B, E1C=E1C), open("./exp7_out/e1c_results.pkl", "wb"))

[L_A level 4] n= 79218 | CE(next | chain,cidx)=1.2238  CE(next | chain,cidx,symbol)=1.2258  gain from symbol=-0.0019 nats
[L_A level 5] n=191334 | CE(next | chain,cidx)=1.2254  CE(next | chain,cidx,symbol)=1.1911  gain from symbol=0.0343 nats
[L_A level 6] n=479560 | CE(next | chain,cidx)=1.0982  CE(next | chain,cidx,symbol)=0.9809  gain from symbol=0.1172 nats
[L_C level 4] n= 79218 | CE(next | chain,cidx)=1.1636  CE(next | chain,cidx,symbol)=1.1561  gain from symbol=0.0075 nats
[L_C level 5] n=191334 | CE(next | chain,cidx)=1.1563  CE(next | chain,cidx,symbol)=1.1477  gain from symbol=0.0086 nats
[L_C level 6] n=479560 | CE(next | chain,cidx)=1.1794  CE(next | chain,cidx,symbol)=1.0502  gain from symbol=0.1292 nats
  ./exp6_bigpool_h4/shared_d96_h4_seed0.pt: vocab=7 d=96 layers=12 heads=4 step=90000 tail-1000 loss=0.7973 langs=['L_A', 'L_B', 'L_C']

[E1b scale | recipient L_A | level 4]  KL(clean||patched) at W=1, nats.   z = (swap-junk)/SE.   recov = swap / full wrong-symbol patch
k

In [ ]:
LEVEL6 = {}
for name in EVAL_MODELS:
    if not os.path.exists(CKPTS[name]["path"]): continue
    model, langs = load_model(CKPTS[name]["path"], CKPTS[name]["n_head"])
    LEVEL6[name] = run_E1c(model, langs, core, tok, nreal,
                           recips=("L_A", "L_B", "L_C"), fit_langs=("L_A", "L_C"),
                           levels=(6,), layers=(2, 3, 4, 5, 6, 7))
    report_E1c(LEVEL6[name], tag=name)
    del model; torch.cuda.empty_cache()
pickle.dump(LEVEL6, open("./exp7_out/e1c_level6.pkl", "wb"))

  ./exp6_bigpool_h4/shared_d96_h4_seed0.pt: vocab=7 d=96 layers=12 heads=4 step=90000 tail-1000 loss=0.7973 langs=['L_A', 'L_B', 'L_C']
  identity patches: max |KL| = 0.00e+00

[bp_h4_d96_s0] fit on L_A | recipient L_A | level 6 | KL at W=1 (nats) | flip = argmax-changed rate
 k r |     C0     Fm    RND reads  flip |  donor:  XN  S_nt [5%,95%]
 2 1 | 0.0000 0.0000 0.0000    NO  0.00 | B: 0.0000   nan (no range / not read) C: 0.0000   nan (no range / not read)
 2 2 | 0.0000 0.0000 0.0000    NO  0.00 | B: 0.0000   nan (no range / not read) C: 0.0000   nan (no range / not read)
 2 3 | 0.0000 0.0000 0.0000    NO  0.00 | B: 0.0000   nan (no range / not read) C: 0.0000   nan (no range / not read)
 3 1 | 0.0000 0.0000 0.0000    NO  0.00 | B: 0.0000   nan (no range / not read) C: 0.0000   nan (no range / not read)
 3 2 | 0.0000 0.0000 0.0000    NO  0.00 | B: 0.0000   nan (no range / not read) C: 0.0000   nan (no range / not read)
 3 3 | 0.0000 0.0000 0.0000    NO  0.00 | B: 0.0000   nan (no ra

In [ ]:
run_E1c(model, langs, core, tok, nreal, recips=("L_A","L_B","L_C"), fit_langs=("L_A","L_C"), levels=(6,), layers=(2,3,4,5,6,7))

NameError: name 'model' is not defined

In [11]:
core = build_eval_core(build_grammar(GRAMMAR_SEED)[0], N_EVAL_TREES, EVAL_SEED)
alignment_selftest(core)
tok, nreal = make_eval_tensors(core)

alignment self-test PASSED (reverse(A)==C, A2==A, spans contiguous, span-start(A)==span-end(C), level-6 A==B)
